# Cross-Market Transfer Learning for Stock Return Prediction

Can a model trained on U.S. stocks help predict next-month stock returns in the United Kingdom and Japan, especially
when little local training data is available? The data are monthly stock-level observations from the
Jensen–Kelly–Pedersen (JKP) Global Factor dataset.

Phase 1 — Start

## 1. Data Loading

The three country files were downloaded from WRDS with the JKP stock screens applied. Only identifiers, screens, size,
the target and the realized return (`ret_1_0`) are loaded here; the remaining characteristics are read in batches when needed.

In [220]:
import os  # settings passed by the Amarel job scripts
import sys  # detect the operating system
import time  # elapsed training time
import warnings  # record model convergence warnings
from itertools import combinations, product  # model pairs and hyperparameter grids
from pathlib import Path  # portable file paths

import joblib  # save trained models
import matplotlib.pyplot as plt  # result figures
import numpy as np  # numerical arrays
import pandas as pd  # tables, dates and Parquet/Excel reading
import pyarrow as pa  # fixed column types for saved predictions
import pyarrow.parquet as pq  # Parquet schemas without loading data
from xgboost import XGBRegressor  # boosted decision trees; must be imported before torch on macOS
import torch  # neural networks, automatic differentiation and GPU support
import torch.nn.functional as F  # efficient attention
from IPython.display import display  # formatted tables in the notebook
from sklearn.exceptions import ConvergenceWarning  # raised when Elastic Net does not converge
from sklearn.linear_model import ElasticNet  # linear benchmark with L1 and L2 penalties
from torch import nn  # neural network layers

In [221]:
# Execution settings: the only place that controls training
#   "off"           define models and functions without training
#   "local_train"   train one Transformer on real data, as set in LOCAL_RUN
#   "full_research" run the experiments listed in RESEARCH_EXPERIMENTS
RUN_MODE = "off"

LOCAL_RUN = {
    "country": "GBR",      # "GBR", "JPN", or "USA" for U.S. pretraining
    "window": "train_1y",  # "train_1y", "train_3y" or "train_5y"; use "train_full" for "USA"
    "seed": 42,
    "max_epochs": 5,       # at most 100
}

RESEARCH_EPOCHS = 100  # maximum Transformer epochs in full research; use 5 for a short laptop run

# Full research, in dependency order; delete entries to run only some of them
RESEARCH_EXPERIMENTS = [
    "elastic_net", "xgboost", "local", "pretrained", "direct", "finetuned", "joint", "localmatched",
]
# To run one experiment: RUN_MODE = "full_research" and, for example, RESEARCH_EXPERIMENTS = ["xgboost"]

RUN_PHASE6_PREDICTIONS = False  # True generates test predictions from the saved models
RUN_FINAL_EVALUATION = False    # True evaluates the saved test predictions

DEVICE_CHOICE = "auto"   # "auto" selects cuda, then mps (Apple GPU), then cpu
RUN_SMOKE_TESTS = False  # True runs small code checks in any mode; they never save models
RUN_PREFLIGHT = False    # True checks that the largest cross-section fits in device memory

In [222]:
ALL_EXPERIMENTS = [
    "elastic_net", "xgboost", "local", "pretrained", "direct", "finetuned", "joint", "localmatched",
]

# Amarel job scripts choose a run with environment variables, so the settings above need no editing there
RUN_MODE = os.environ.get("CMP_RUN_MODE", RUN_MODE)
RESEARCH_EPOCHS = int(os.environ.get("CMP_EPOCHS", RESEARCH_EPOCHS))
RESEARCH_EXPERIMENTS = os.environ.get("CMP_EXPERIMENTS", ",".join(RESEARCH_EXPERIMENTS)).split(",")
DEVICE_CHOICE = os.environ.get("CMP_DEVICE", DEVICE_CHOICE)


def switch(name, default):
    """Read an on/off environment variable ("1" or "0")."""
    return {"1": True, "0": False}[os.environ[name]] if name in os.environ else default


RUN_PHASE6_PREDICTIONS = switch("CMP_PREDICTIONS", RUN_PHASE6_PREDICTIONS)
RUN_FINAL_EVALUATION = switch("CMP_EVALUATION", RUN_FINAL_EVALUATION)
RUN_SMOKE_TESTS = switch("CMP_SMOKE_TESTS", RUN_SMOKE_TESTS)
RUN_PREFLIGHT = switch("CMP_PREFLIGHT", RUN_PREFLIGHT)

if RUN_MODE not in ["off", "local_train", "full_research"]:
    raise ValueError(f"Unknown RUN_MODE: {RUN_MODE}")
if not 1 <= RESEARCH_EPOCHS <= 100:
    raise ValueError("RESEARCH_EPOCHS must be between 1 and 100.")
if not set(RESEARCH_EXPERIMENTS) <= set(ALL_EXPERIMENTS):
    raise ValueError(f"Unknown experiments: {set(RESEARCH_EXPERIMENTS) - set(ALL_EXPERIMENTS)}")


def should_run(experiment):
    """True if the experiment is part of the current full-research run."""
    if RUN_MODE == "full_research" and experiment in RESEARCH_EXPERIMENTS:
        return True
    reason = f"RUN_MODE = '{RUN_MODE}'" if RUN_MODE != "full_research" else "not in RESEARCH_EXPERIMENTS"
    print(f"{experiment}: not run ({reason})")
    return False


if sys.platform == "darwin":
    torch.set_num_threads(1)  # avoids a macOS crash between the PyTorch and XGBoost thread libraries (CPU only)

print(f"Run mode: {RUN_MODE} | maximum epochs: {RESEARCH_EPOCHS} | predictions: {RUN_PHASE6_PREDICTIONS} | "
      f"evaluation: {RUN_FINAL_EVALUATION} | smoke tests: {RUN_SMOKE_TESTS} | preflight: {RUN_PREFLIGHT}")

Run mode: off


In [223]:
pd.set_option("display.float_format", "{:,.4f}".format)

# Paths are relative to the project folder; the environment variables are optional overrides
PROJECT_ROOT = Path(os.environ.get("CMP_PROJECT_ROOT", "."))
DATA_DIR = Path(os.environ.get("CMP_DATA_DIR", PROJECT_ROOT / "data"))

COUNTRIES = ["USA", "GBR", "JPN"]
FILES = {country: DATA_DIR / f"{country}.parquet" for country in COUNTRIES}

TARGET = "ret_exc_lead1m"
SCREENS = ["common", "exch_main", "primary_sec", "obs_main"]
METADATA_COLS = ["id", "eom", "excntry", "gvkey", "permno", "size_grp", "me", TARGET] + SCREENS

for path in FILES.values():
    if not path.exists():
        raise FileNotFoundError(f"Missing dataset: {path}. See data/README.md.")

In [ ]:
print("Loading JKP datasets...")

data = {}
for country, path in FILES.items():
    df = pd.read_parquet(path, columns=METADATA_COLS + ["ret_1_0"])
    df["eom"] = pd.to_datetime(df["eom"])
    for column in ["excntry", "gvkey", "size_grp"]:
        df[column] = df[column].astype("category")  # saves memory
    data[country] = df.sort_values(["id", "eom"], ignore_index=True)

display(pd.concat([df.head(2) for df in data.values()], ignore_index=True).drop(columns=SCREENS))

## 2. Dataset Structure

The model inputs come from the 153 characteristics JKP use for published factors, listed in the `abr_jkp` field of the
[official metadata](https://jkpfactors.com/jkp-wrds-guide).

In [225]:
FACTOR_URL = "https://github.com/bkelly-lab/jkp-data/raw/main/src/jkp/data/resources/factor_details.xlsx"

schemas = {country: pq.read_schema(path) for country, path in FILES.items()}
try:
    characteristics = pd.read_excel(FACTOR_URL)["abr_jkp"].dropna().drop_duplicates().tolist()
except OSError as error:  # includes network errors and timeouts
    characteristics = [c for c in schemas["USA"].names if c not in METADATA_COLS]
    print(f"WARNING: official metadata unavailable ({error}); characteristic names are provisional.")

if len(characteristics) != 153:
    raise ValueError(f"Expected 153 characteristics, found {len(characteristics)}.")
for country, schema in schemas.items():
    missing = [c for c in METADATA_COLS + characteristics if c not in schema.names]
    if missing:
        raise ValueError(f"{country}: missing columns {missing}")
    if not schema.equals(schemas["USA"]):
        raise ValueError(f"{country}: column names or types differ from USA.")

In [226]:
structure = pd.DataFrame({
    country: {
        "Observations": f"{len(df):,}", "Unique stocks": f"{df['id'].nunique():,}",
        "Columns": len(schemas[country].names),
        "First month": df["eom"].min().strftime("%Y-%m"), "Last month": df["eom"].max().strftime("%Y-%m"),
    } for country, df in data.items()
}).T
display(structure)
print(f"All {len(characteristics)} characteristics and {len(METADATA_COLS)} other columns are present, with identical schemas.")

,Observations,Unique stocks,Columns,First month,Last month
USA,"1,604,094","18,733",165,2000-01,2025-12
GBR,"534,295","5,182",165,2000-01,2025-12
JPN,"1,175,157","6,299",165,2000-01,2025-12


All 153 characteristics and 12 other columns are present, with identical schemas.


The proposal mentions 174 retained columns; the files contain 165, which include every field the proposal names.

## 3. Data Integrity Checks

The characteristics are scanned once, in batches, for missing and infinite values. All counts in the table should be zero.

In [227]:
print("Checking dataset integrity...")

missing_pct = pd.DataFrame(index=characteristics, columns=COUNTRIES, dtype=float)
infinite_values = dict.fromkeys(COUNTRIES, 0)

for country, path in FILES.items():
    for start in range(0, len(characteristics), 20):
        batch = pd.read_parquet(path, columns=characteristics[start:start + 20])
        missing_pct.loc[batch.columns, country] = 100 * batch.isna().mean()
        infinite_values[country] += int(np.isinf(batch.to_numpy()).sum())
    infinite_values[country] += int(np.isinf(data[country][["me", TARGET]].to_numpy()).sum())

Checking dataset integrity...


In [228]:
calendar = pd.date_range("2000-01-31", max(df["eom"].max() for df in data.values()), freq="ME")

integrity = pd.DataFrame({
    country: {
        "Wrong country code": (df["excntry"] != country).sum(),
        "Missing id or eom": df[["id", "eom"]].isna().any(axis=1).sum(),
        "Duplicate keys": df.duplicated(["excntry", "id", "eom"]).sum(),
        "Screen violations": (df[SCREENS] != 1).any(axis=1).sum(),
        "Missing months": len(calendar.difference(df["eom"])),
        "Infinite values": infinite_values[country],
    } for country, df in data.items()
})
display(integrity)

if integrity.to_numpy().sum() > 0:
    raise ValueError("Integrity check failed; see the nonzero counts above.")
print("All integrity checks passed.")

,USA,GBR,JPN
Wrong country code,0,0,0
Missing id or eom,0,0,0
Duplicate keys,0,0,0
Screen violations,0,0,0
Missing months,0,0,0
Infinite values,0,0,0


All integrity checks passed.


The target should equal the stock's realized return in the next month, `ret_1_0`, minus the T-bill rate. The share of
consecutive-month pairs agreeing within one percentage point checks this timing within the JKP data.

In [229]:
for country, df in data.items():
    by_stock = df.groupby("id")
    next_return = by_stock["ret_1_0"].shift(-1)
    consecutive = by_stock["eom"].shift(-1) == df["eom"] + pd.offsets.MonthEnd(1)
    difference = (next_return - df[TARGET])[consecutive].dropna()
    agreement = (difference.abs() <= 0.01).mean()
    print(f"{country}: {agreement:.2%} of {len(difference):,} targets match the next month's return")
    if agreement < 0.99:
        raise ValueError(f"{country}: the target is not aligned with the following month.")

USA: 100.00% of 1,545,608 targets match the next month's return
GBR: 100.00% of 523,283 targets match the next month's return
JPN: 100.00% of 1,168,162 targets match the next month's return


## 4. Missing Values and Target Availability

Values are reported as they are; nothing is filled, clipped or removed. Returns are in decimal units (1.0 = 100%).

In [230]:
print("Checking missing values...")

missing_summary = pd.DataFrame({
    country: {
        "Missing target (%)": 100 * df[TARGET].isna().mean(),
        "Missing size group (%)": 100 * df["size_grp"].isna().mean(),
        "Mean predictor missing (%)": missing_pct[country].mean(),
        "Median predictor missing (%)": missing_pct[country].median(),
        "Predictors over 50% missing": (missing_pct[country] > 50).sum(),
    } for country, df in data.items()
})
display(missing_summary)

print("Most incomplete characteristics (% missing, ranked by the three-country average):")
most_missing = missing_pct.mean(axis=1).nlargest(10).index
display(missing_pct.loc[most_missing])

Checking missing values...


,USA,GBR,JPN
Missing target (%),3.5689,1.8346,0.2596
Missing size group (%),0.4035,0.4469,0.2017
Mean predictor missing (%),14.7485,31.7971,26.1489
Median predictor missing (%),8.8561,29.4992,19.6708
Predictors over 50% missing,9.0000,20.0000,17.0000


Most incomplete characteristics (% missing, ranked by the three-country average):


,USA,GBR,JPN
seas_16_20na,72.8430,86.8644,64.4921
seas_16_20an,72.6464,85.4513,64.0935
rd5_at,61.4115,84.7575,62.7640
seas_11_15na,63.1121,76.4520,46.8341
seas_11_15an,62.9113,73.8515,46.3072
saleq_su,17.2693,64.6815,99.3574
rd_sale,53.0261,75.2195,51.7802
rd_me,50.3900,74.4259,51.7704
ocfq_saleq_std,19.5343,57.5884,99.3533
ni_inc8q,19.0711,48.3154,99.2096


In [231]:
print("Target distribution (decimal returns):")
display(pd.DataFrame({
    country: df[TARGET].describe(percentiles=[0.01, 0.5, 0.99, 0.999]) for country, df in data.items()
}))

print("Available targets near the dataset endpoint:")
endpoint = pd.DataFrame({
    country: df[df["eom"] >= "2025-09-30"].groupby("eom")[TARGET].count() for country, df in data.items()
})
endpoint.insert(0, "predicted month", (endpoint.index + pd.offsets.MonthEnd(1)).strftime("%Y-%m"))
display(endpoint)

Target distribution (decimal returns):


,USA,GBR,JPN
count,"1,546,846.0000","524,493.0000","1,172,106.0000"
mean,0.0253,0.0045,0.0066
std,13.3882,0.6662,0.1361
min,-1.0050,-1.0022,-0.9993
1%,-0.4670,-0.4440,-0.2721
50%,-0.0001,-0.0061,-0.0019
99%,0.6393,0.6303,0.4244
99.9%,1.8455,1.8495,1.1178
max,"15,348.9993",331.0520,11.9461


Available targets near the dataset endpoint:


,predicted month,USA,GBR,JPN
eom,,,,
2025-09-30,2025-10,4808,1185,3944
2025-10-31,2025-11,4806,1185,3944
2025-11-30,2025-12,4813,1187,3941
2025-12-31,2026-01,4,1181,3942


Some targets are extremely large, and squared-error models are sensitive to them. The data remain unchanged here; the
treatment must be decided before model training. The U.S. has almost no January 2026 targets, so the common test
period ends in December 2025.

## 5. Modeling Data Requirements

The target `ret_exc_lead1m` is the following month's USD stock return in excess of the U.S. T-bill rate. Periods are
defined by the **predicted-return month**, which is one month after the observation month `eom`.

| Period | Observation months (`eom`) | Predicted-return months |
|---|---|---|
| U.S. training | Jan 2000 – Nov 2017 | Feb 2000 – Dec 2017 |
| Local training (1, 3, 5 years) | from Dec 2016, Dec 2014, Dec 2012 – Nov 2017 | from Jan 2017, Jan 2015, Jan 2013 – Dec 2017 |
| Validation | Dec 2017 – Nov 2019 | Jan 2018 – Dec 2019 |
| Testing | Dec 2019 – Nov 2025 | Jan 2020 – Dec 2025 (72 months) |

Phase 1 — End

Phase 2 — Start

## 6. Temporal Data Splitting

Periods are defined by the predicted-return month, which is one month after `eom`. Local training windows of 1, 3 and
5 years all end in December 2017. Observations are assigned by date only, never at random.

In [232]:
# First and last predicted-return month of each period
PERIODS = {
    "train_full": ("2000-02", "2017-12"),
    "train_5y": ("2013-01", "2017-12"),
    "train_3y": ("2015-01", "2017-12"),
    "train_1y": ("2017-01", "2017-12"),
    "validation": ("2018-01", "2019-12"),
    "test": ("2020-01", "2025-12"),
}
PERIODS = {
    name: (pd.Timestamp(first) + pd.offsets.MonthEnd(0), pd.Timestamp(last) + pd.offsets.MonthEnd(0))
    for name, (first, last) in PERIODS.items()
}
TRAIN_WINDOWS = {
    "USA": ["train_full"],
    "GBR": ["train_1y", "train_3y", "train_5y"],
    "JPN": ["train_1y", "train_3y", "train_5y"],
}

for df in data.values():
    df["predicted_month"] = df["eom"] + pd.offsets.MonthEnd(1)


def in_period(predicted_month, period):
    """True where the predicted-return month belongs to the period."""
    return predicted_month.between(*PERIODS[period])

In [233]:
print("Preparing chronological data splits...")

rows = []
for country, df in data.items():
    for period in TRAIN_WINDOWS[country] + ["validation", "test"]:
        part = df[in_period(df["predicted_month"], period)]
        rows.append({
            "Country": country, "Period": period, "Observations": len(part),
            "Labeled": part[TARGET].notna().sum(), "Months": part["eom"].nunique(),
            "First eom": part["eom"].min(), "Last eom": part["eom"].max(),
        })
split_summary = pd.DataFrame(rows)
display(split_summary)

Preparing chronological data splits...


,Country,Period,Observations,Labeled,Months,First eom,Last eom
0,USA,train_full,1129349,1091387,215,2000-01-31,2017-11-30
1,USA,validation,106502,101787,24,2017-12-31,2019-11-30
2,USA,test,363367,353668,72,2019-12-31,2025-11-30
3,GBR,train_1y,18606,18239,12,2016-12-31,2017-11-30
4,GBR,train_3y,57465,56515,36,2014-12-31,2017-11-30
5,GBR,train_5y,96737,95227,60,2012-12-31,2017-11-30
6,GBR,validation,36808,36378,24,2017-12-31,2019-11-30
7,GBR,test,100297,99377,72,2019-12-31,2025-11-30
8,JPN,train_1y,44705,44650,12,2016-12-31,2017-11-30
9,JPN,train_3y,132899,132686,36,2014-12-31,2017-11-30


In [234]:
expected_months = {name: len(pd.date_range(*dates, freq="ME")) for name, dates in PERIODS.items()}
assert (split_summary["Months"] == split_summary["Period"].map(expected_months)).all(), "A period has missing months."

# Training ends before validation, and validation ends before testing
train_names = [name for name in PERIODS if name.startswith("train")]
assert {PERIODS[name][1] for name in train_names} == {pd.Timestamp("2017-12-31")}
assert PERIODS["train_full"][1] < PERIODS["validation"][0]
assert PERIODS["validation"][1] < PERIODS["test"][0]
print("All periods are complete and do not overlap; all training windows end in December 2017.")

All periods are complete and do not overlap; all training windows end in December 2017.


## 7. Predictor Selection

One common predictor set is used by all models. A characteristic is kept if, in **each** of the seven training windows
(U.S. full history; UK and Japan 1, 3 and 5 years), it is available for at least 50% of observations and takes at least
two distinct values. The denominator is the number of **labeled** training observations, because only these enter
supervised training. Infinite values count as unavailable.

In [235]:
MIN_COVERAGE = 0.50


def read_observations(country, period, columns):
    """Read selected columns for one period, ordered by month and stock."""
    first, last = [date - pd.offsets.MonthEnd(1) for date in PERIODS[period]]  # predicted month to eom
    return read_months(country, first, last, columns)


def read_months(country, first_eom, last_eom, columns):
    """Read selected columns for a range of observation months, ordered by month and stock."""
    dates = [("eom", ">=", first_eom.date()), ("eom", "<=", last_eom.date())]
    df = pd.read_parquet(FILES[country], columns=["id", "eom"] + columns, filters=dates)
    df["eom"] = pd.to_datetime(df["eom"])
    return df.sort_values(["eom", "id"], ignore_index=True)

In [236]:
print("Measuring training coverage and variation...")

coverage, variation, selection_last_eom = {}, {}, {}
for country, windows in TRAIN_WINDOWS.items():
    longest = windows[-1]  # the shorter windows are subsets of this one
    for start in range(0, len(characteristics), 20):
        names = characteristics[start:start + 20]
        df = read_observations(country, longest, [TARGET] + names)
        df = df[df[TARGET].notna()]
        selection_last_eom[country] = df["eom"].max()
        for window in windows:
            part = df.loc[in_period(df["eom"] + pd.offsets.MonthEnd(1), window), names]
            part = part.where(np.isfinite(part))
            for name in names:
                coverage.setdefault(f"{country} {window}", {})[name] = part[name].notna().mean()
                variation.setdefault(f"{country} {window}", {})[name] = part[name].min() < part[name].max()

coverage = pd.DataFrame(coverage).loc[characteristics]
variation = pd.DataFrame(variation).loc[characteristics]

Measuring training coverage and variation...


In [237]:
enough_coverage = (coverage >= MIN_COVERAGE).all(axis=1)
enough_variation = variation.all(axis=1)

# A characteristic failing both rules is counted once, under coverage
feature_table = coverage.assign(status="selected")
feature_table.loc[~enough_variation, "status"] = "insufficient variation"
feature_table.loc[~enough_coverage, "status"] = "insufficient coverage"

selected_features = feature_table.index[feature_table["status"] == "selected"].tolist()
assert not set(selected_features) & set(METADATA_COLS), "Metadata columns cannot be predictors."

print(f"Initial characteristics:          {len(characteristics)}")
print(f"Removed, insufficient coverage:   {(feature_table['status'] == 'insufficient coverage').sum()}")
print(f"Removed, insufficient variation:  {(feature_table['status'] == 'insufficient variation').sum()}")
print(f"Selected predictors:              {len(selected_features)}")
print()
for start in range(0, len(selected_features), 10):
    print(", ".join(selected_features[start:start + 10]))

Initial characteristics:          153
Removed, insufficient coverage:   27
Removed, insufficient variation:  0
Selected predictors:              126

ret_6_1, ret_12_1, tax_gr1a, prc_highprc_252d, resff3_6_1, resff3_12_1, be_me, debt_me, at_me, ret_60_12
ni_me, fcf_me, sale_gr3, sale_gr1, ebitda_mev, sale_me, ocf_me, bev_mev, netdebt_me, eq_dur
capex_abn, at_gr1, ppeinv_gr1a, noa_at, noa_gr1a, lnoa_gr1a, capx_gr1, capx_gr2, capx_gr3, chcsho_12m
eqnpo_12m, debt_gr3, inv_gr1, inv_gr1a, oaccruals_at, taccruals_at, cowc_gr1a, coa_gr1a, col_gr1a, nncoa_gr1a
ncoa_gr1a, ncol_gr1a, nfna_gr1a, sti_gr1a, lti_gr1a, fnl_gr1a, be_gr1a, oaccruals_ni, taccruals_ni, netis_at
eqnetis_at, dbnetis_at, ebit_bev, ebit_sale, sale_bev, at_turnover, gp_at, gp_atl1, ope_be, ope_bel1
op_at, op_atl1, cop_at, cop_atl1, f_score, o_score, z_score, pi_nix, at_be, opex_at
emp_gr1, age, dsale_dinv, dsale_drec, dgp_dsale, dsale_dsga, sale_emp_gr1, tangibility, cash_at, ni_ar1
ni_ivol, earnings_variability, aliq_at, ali

In [238]:
excluded = feature_table[feature_table["status"] != "selected"].copy()
excluded.insert(0, "lowest coverage", excluded.drop(columns="status").min(axis=1))
excluded.insert(1, "lowest in", excluded.drop(columns=["status", "lowest coverage"]).idxmin(axis=1))
print("Excluded characteristics (coverage as a share of labeled training observations):")
display(excluded[["status", "lowest coverage", "lowest in"]].sort_values("lowest coverage"))

Excluded characteristics (coverage as a share of labeled training observations):


,status,lowest coverage,lowest in
ivol_hxz4_21d,insufficient coverage,0.0000,JPN train_1y
iskew_hxz4_21d,insufficient coverage,0.0000,JPN train_1y
ocfq_saleq_std,insufficient coverage,0.0081,JPN train_1y
saleq_su,insufficient coverage,0.0083,JPN train_3y
saleq_gr1,insufficient coverage,0.0092,JPN train_3y
ni_inc8q,insufficient coverage,0.0110,JPN train_1y
niq_su,insufficient coverage,0.0116,JPN train_5y
niq_be_chg1,insufficient coverage,0.0121,JPN train_1y
niq_at_chg1,insufficient coverage,0.0122,JPN train_1y
niq_be,insufficient coverage,0.0130,JPN train_5y


## 8. Cross-Sectional Normalization

Within each country-month, every predictor is ranked across all screened stocks, including stocks without a return
label, and mapped to [−0.5, 0.5] by `(rank − 1) / (n − 1) − 0.5`. Ties receive average ranks. A single available
observation receives zero, and missing values stay missing until the next section.

In [239]:
def rank_within_month(features, months):
    """Rank each column within each month and map the ranks to [-0.5, 0.5]."""
    values = features.where(np.isfinite(features))
    by_month = values.groupby(months)
    ranks = by_month.rank(method="average")
    n = by_month.transform("count")
    scaled = (ranks - 1) / (n - 1) - 0.5
    return scaled.mask((n == 1) & values.notna(), 0.0)

In [ ]:
example_month = pd.Timestamp("2017-06-30")
rows = []
for country in COUNTRIES:
    raw = read_months(country, example_month, example_month, selected_features)
    ranked = rank_within_month(raw[selected_features], raw["eom"])

    # Show the smallest, middle and largest value, and one missing value, of one incomplete predictor
    feature = raw[selected_features].isna().mean().idxmax()
    order = raw[feature].dropna().sort_values().index
    picks = [order[0], order[len(order) // 2], order[-1], raw.index[raw[feature].isna()][0]]
    for i in picks:
        rows.append({
            "Country": country, "id": raw.loc[i, "id"], "Characteristic": feature,
            "Original": raw.loc[i, feature], "Ranked": ranked.loc[i, feature],
        })
print(f"Ranking example, {example_month:%B %Y}:")
display(pd.DataFrame(rows))

## 9. Missing Data Handling

Missing ranked predictors receive zero, the neutral point of the scale. Missing returns are never filled; a label mask
marks the stocks that can be used in losses and metrics. Finite target values are kept unchanged.

In [241]:
feature_types = {str(schemas["USA"].field(name).type) for name in selected_features}
print(f"Data types of the selected predictors: {feature_types}")
assert feature_types == {"double"}, "A selected predictor is not numeric and needs categorical handling."


def prepare_features(features, months):
    """Rank within month, then replace missing ranks with the neutral value zero."""
    return rank_within_month(features, months).fillna(0.0).astype("float32")

Data types of the selected predictors: {'double'}


All selected predictors are numeric, so no categorical treatment is needed.

## 10. Model Input Preparation

`prepare_data` returns two aligned objects for one country and one period: `info` (identifiers, months, target and
label mask) and `X` (a `float32` matrix with columns ordered as `selected_features`). Data are read one year at a time,
so raw characteristics for a full history are never in memory together. `month_labels`, the number of labeled stocks in
the month, allows equal monthly weighting during training.

In [242]:
def prepare_data(country, period, features):
    """Prepare one country and period; returns (info, X) ordered by month and stock."""
    first, last = [date - pd.offsets.MonthEnd(1) for date in PERIODS[period]]  # predicted month to eom
    return prepare_months(country, first, last, features)


def prepare_months(country, first_eom, last_eom, features):
    """Prepare a range of observation months; returns (info, X) ordered by month and stock."""
    n_rows = data[country]["eom"].between(first_eom, last_eom).sum()
    X = np.empty((n_rows, len(features)), dtype="float32")
    info, position = [], 0

    for year in range(first_eom.year, last_eom.year + 1):
        start = max(first_eom, pd.Timestamp(year, 1, 31))
        end = min(last_eom, pd.Timestamp(year, 12, 31))
        df = read_months(country, start, end, [TARGET] + features)
        X[position:position + len(df)] = prepare_features(df[features], df["eom"]).to_numpy()
        position += len(df)
        info.append(df[["id", "eom", TARGET]])

    assert position == n_rows, "Prepared rows do not match the dataset."
    info = pd.concat(info, ignore_index=True)
    info.insert(0, "country", country)
    info["predicted_month"] = info["eom"] + pd.offsets.MonthEnd(1)
    info["has_label"] = np.isfinite(info[TARGET])
    info["month_labels"] = info.groupby("eom")["has_label"].transform("sum")
    return info, X


def get_cross_section(info, X, eom):
    """Return all stocks of one observation month."""
    rows = (info["eom"] == eom).to_numpy()
    return info[rows], X[rows]

In [ ]:
print("Preparing example cross-sections...")

EXAMPLE_MONTHS = {"training": "2017-11-30", "validation": "2018-12-31", "testing": "2025-11-30"}
examples, rows = {}, []
for country in COUNTRIES:
    for period, eom in EXAMPLE_MONTHS.items():
        eom = pd.Timestamp(eom)
        info, X = prepare_months(country, eom - pd.offsets.MonthEnd(1), eom, selected_features)  # two months
        examples[country, period] = (info, X)
        month_info, month_X = get_cross_section(info, X, eom)
        rows.append({
            "Country": country, "Period": period, "eom": eom, "Stocks": len(month_info),
            "Predictors": month_X.shape[1], "Labels": month_info["has_label"].sum(),
            "Missing labels": (~month_info["has_label"]).sum(), "X shape": month_X.shape,
        })
display(pd.DataFrame(rows))
display(info.head(3))

## 11. Preprocessing Verification

The checks below use the prepared examples and a small artificial table for ranking edge cases.

In [244]:
# Selection used training months only
assert all(last == pd.Timestamp("2017-11-30") for last in selection_last_eom.values())

# Ranking rules: ties, missing and infinite values, single observation, empty month, independent months
test_months = pd.Series([1, 1, 1, 1, 2, 2, 3])
test_values = pd.DataFrame({"x": [5.0, 1.0, 5.0, np.inf, 9.0, np.nan, np.nan]})
expected = pd.DataFrame({"x": [0.25, -0.5, 0.25, np.nan, 0.0, np.nan, np.nan]})
pd.testing.assert_frame_equal(rank_within_month(test_values, test_months), expected)
assert (prepare_features(test_values, test_months)["x"].iloc[[3, 5, 6]] == 0).all()
print("Selection dates and ranking rules verified.")

Selection dates and ranking rules verified.


In [245]:
for (country, period), (info, X) in examples.items():
    raw = read_months(country, info["eom"].min(), info["eom"].max(), [TARGET] + selected_features)
    original = data[country].set_index(["id", "eom"])[TARGET].loc[pd.MultiIndex.from_frame(info[["id", "eom"]])]

    assert X.dtype == "float32" and X.shape == (len(info), len(selected_features))
    assert np.isfinite(X).all() and X.min() >= -0.5 and X.max() <= 0.5
    assert (X[raw[selected_features].isna().to_numpy()] == 0).all()  # missing predictors are zero
    assert raw[["id", "eom"]].equals(info[["id", "eom"]])  # identifiers preserved
    np.testing.assert_array_equal(info[TARGET], original)  # targets unchanged, missing stays missing
    assert (info["has_label"] == info[TARGET].notna()).all()
    assert (info["predicted_month"] == info["eom"] + pd.offsets.MonthEnd(1)).all()

assert all(schema.equals(schemas["USA"]) for schema in (pq.read_schema(path) for path in FILES.values()))
assert all(pq.read_metadata(FILES[c]).num_rows == len(data[c]) for c in COUNTRIES)
print(f"Verified {len(examples)} prepared samples: predictors, identifiers, targets and label masks are correct.")
print("Raw Parquet files keep the same schema and row counts.")

Verified 9 prepared samples: predictors, identifiers, targets and label masks are correct.
Raw Parquet files keep the same schema and row counts.


**Return persistence.** Pooled correlation between a stock's excess returns in consecutive months, using only returns
through December 2017.

In [246]:
rows = []
for country, df in data.items():
    by_stock = df.groupby("id")  # data are sorted by id and eom
    pairs = pd.DataFrame({"previous": by_stock[TARGET].shift(1), "current": df[TARGET]})
    consecutive = df["eom"] == by_stock["eom"].shift(1) + pd.offsets.MonthEnd(1)
    pairs = pairs[consecutive & in_period(df["predicted_month"], "train_full")].dropna()
    rows.append({
        "Country": country, "Pairs": len(pairs), "Pearson": pairs.corr().iloc[0, 1],
        "Spearman": pairs.rank().corr().iloc[0, 1],  # correlation of ranks
    })
display(pd.DataFrame(rows))

,Country,Pairs,Pearson,Spearman
0,USA,1078361,-0.0001,0.0094
1,GBR,379562,0.0041,0.0723
2,JPN,788268,0.0081,0.0029


Phase 2 — End

Phase 3 — Start

## 12. Baseline Model Configuration

Elastic Net and XGBoost predict a stock's next-month excess return from its own characteristics. Each is trained
separately for the UK and Japan on 1, 3 and 5 years of local data, giving six configurations per model family.

Training is controlled by `RUN_MODE` at the top of the notebook. The optional smoke test only checks that the code works
on a few months; its numbers are not research results.

In [247]:
SEED = 42
N_JOBS = int(os.environ.get("SLURM_CPUS_PER_TASK", 4))  # CPU threads for XGBoost
MAX_SETTINGS = 20  # proposal limit per model family and configuration

LOCAL_COUNTRIES = ["GBR", "JPN"]
# Shortened runs have their own folder, so they are never mistaken for the 100-epoch results
RESULTS_DIR = PROJECT_ROOT / ("results" if RESEARCH_EPOCHS == 100 else f"results_{RESEARCH_EPOCHS}epochs")
MODELS_DIR = RESULTS_DIR / "models"
VALIDATION_FILE = RESULTS_DIR / "baseline_validation.csv"


def save_safely(path, save):
    """Write to a temporary file first, so an interrupted job never leaves a partial file."""
    temporary = path.with_name(path.name + ".tmp")
    save(temporary)
    temporary.replace(path)

Every training month has the same total weight: a stock's weight is proportional to one over the number of labeled
stocks in its month, scaled to average one. Validation MSE is the average of the monthly MSE values.

In [248]:
def keep_labeled(info, X):
    """Keep observations with an available return; they are the only ones used in losses."""
    labeled = info["has_label"].to_numpy()
    return {"info": info[labeled].reset_index(drop=True), "X": X[labeled], "y": info.loc[labeled, TARGET].to_numpy()}


def load_labeled(country, period):
    """Load labeled training or validation observations."""
    if period == "test":
        raise ValueError("Test data cannot be used for training or model selection.")
    return keep_labeled(*prepare_data(country, period, selected_features))


def month_weights(info):
    """Weights that give each month equal total weight and average one."""
    month_size = info.groupby("eom")["id"].transform("size").to_numpy()
    return len(info) / (info["eom"].nunique() * month_size)


def monthly_mse(info, y, prediction):
    """Average of monthly mean squared errors."""
    squared_error = pd.Series((y - prediction) ** 2)
    return squared_error.groupby(info["eom"].to_numpy()).mean().mean()

In [249]:
def fit_model(model, train):
    """Fit with monthly-equal weights; returns True if no convergence warning was raised."""
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        model.fit(train["X"], train["y"], sample_weight=month_weights(train["info"]))
    for warning in caught:
        if not issubclass(warning.category, ConvergenceWarning):
            warnings.warn_explicit(warning.message, warning.category, warning.filename, warning.lineno)
    return not any(issubclass(warning.category, ConvergenceWarning) for warning in caught)


def predict_model(model, sample):
    """Predictions aligned with stock identifiers and months."""
    predictions = sample["info"][["country", "id", "eom", "predicted_month"]].copy()
    predictions["prediction"] = model.predict(sample["X"])
    return predictions

In [250]:
if RUN_SMOKE_TESTS:
    # Last three UK training months, and the first three validation months to check the evaluation code
    month_end = pd.offsets.MonthEnd
    train_end, validation_end = [PERIODS[p][1] - month_end(1) for p in ("train_1y", "validation")]
    validation_start = PERIODS["validation"][0] - month_end(1)
    smoke_train = keep_labeled(*prepare_months("GBR", train_end - month_end(2), train_end, selected_features))
    smoke_valid = keep_labeled(
        *prepare_months("GBR", validation_start, validation_start + month_end(2), selected_features)
    )
    print(f"Smoke-test training sample:   {smoke_train['X'].shape}")
    print(f"Smoke-test validation sample: {smoke_valid['X'].shape}")

## 13. Elastic Net Regression

Squared error with L1 and L2 coefficient penalties. `alpha` sets the overall penalty and `l1_ratio` the share of L1.
The intercept is not penalized. Predictors already share a common scale, so no further scaling is applied.

In [251]:
ELASTIC_NET_GRID = [
    {"alpha": alpha, "l1_ratio": l1_ratio}
    for alpha, l1_ratio in product([1e-5, 1e-4, 3e-4, 1e-3, 1e-2], [0.1, 0.4, 0.7, 0.95])
]


def make_elastic_net(params):
    # precompute=True uses the 126 x 126 Gram matrix, which is faster for many rows
    return ElasticNet(**params, precompute=True, max_iter=100_000, tol=1e-4, random_state=SEED)


print(f"Elastic Net settings: {len(ELASTIC_NET_GRID)}")

Elastic Net settings: 20


In [252]:
if RUN_SMOKE_TESTS:
    smoke_models = {"Elastic Net": make_elastic_net({"alpha": 1e-3, "l1_ratio": 0.5})}
    converged = fit_model(smoke_models["Elastic Net"], smoke_train)
    print(f"Elastic Net smoke test: converged = {converged}, "
          f"nonzero coefficients = {(smoke_models['Elastic Net'].coef_ != 0).sum()}")

## 14. XGBoost Regression

Boosted trees with squared-error loss. Tree depth, learning rate and number of trees are set by an explicit grid, so
the number of trees is chosen by the same monthly-equal validation MSE as every other setting.

In [253]:
XGBOOST_GRID = [
    {"max_depth": depth, "learning_rate": rate, "n_estimators": trees}
    for depth, rate, trees in product([2, 4, 6], [0.01, 0.03, 0.1], [100, 300])
]


def make_xgboost(params):
    return XGBRegressor(**params, objective="reg:squarederror", tree_method="hist", n_jobs=N_JOBS, random_state=SEED)


print(f"XGBoost settings: {len(XGBOOST_GRID)}")

XGBoost settings: 18


In [254]:
if RUN_SMOKE_TESTS:
    smoke_models["XGBoost"] = make_xgboost({"max_depth": 2, "learning_rate": 0.1, "n_estimators": 20})
    fit_model(smoke_models["XGBoost"], smoke_train)

    rows = []
    for family, model in smoke_models.items():
        prediction = predict_model(model, smoke_valid)["prediction"].to_numpy()
        rows.append({
            "Model": family, "Training shape": smoke_train["X"].shape, "Prediction shape": prediction.shape,
            "All finite": np.isfinite(prediction).all(),
            "Monthly MSE (code check only)": monthly_mse(smoke_valid["info"], smoke_valid["y"], prediction),
        })
    display(pd.DataFrame(rows))

## 15. Validation and Model Selection

For each country, training history and model family, every setting is fitted on the local training window and scored on
the 2018–2019 validation period. The setting with the lowest monthly-equal MSE is kept, and is not refitted on
validation data. The zero-forecast MSE and the largest absolute prediction are recorded to reveal distortion from
extreme returns. A prediction standard deviation of zero means every stock received the same forecast.

In [255]:
MODEL_FAMILIES = {
    "Elastic Net": (make_elastic_net, ELASTIC_NET_GRID),
    "XGBoost": (make_xgboost, XGBOOST_GRID),
}


def search(family, train, valid):
    """Evaluate every setting; returns the best model and a table of all settings."""
    make_model, grid = MODEL_FAMILIES[family]
    best_model, best_mse, trials = None, np.inf, []
    started = time.time()
    for number, params in enumerate(grid, start=1):
        model = make_model(params)
        converged = fit_model(model, train)
        prediction = model.predict(valid["X"])
        mse = monthly_mse(valid["info"], valid["y"], prediction)
        trials.append({
            **params, "validation_mse": mse, "max_abs_prediction": np.abs(prediction).max(),
            "prediction_std": prediction.std(), "converged": converged,
        })
        if np.isfinite(mse) and mse < best_mse:
            best_model, best_mse = model, mse
        print(f"  setting {number}/{len(grid)} {params} | validation {mse:.6f} | best {best_mse:.6f} | "
              f"{time.time() - started:.0f}s", flush=True)
    if best_model is None:
        raise ValueError(f"{family}: no setting produced a finite validation MSE.")

    trials = pd.DataFrame(trials)
    trials["selected"] = trials.index == trials["validation_mse"].idxmin()
    return best_model, trials

In [256]:
BASELINE_EXPERIMENTS = {"elastic_net": "Elastic Net", "xgboost": "XGBoost"}


def baseline_path(country, window, experiment):
    return MODELS_DIR / f"{country}_{window}_{experiment}.joblib"


def baseline_complete(country, window, experiment):
    """True if the saved model uses the current predictors and its results are recorded."""
    path = baseline_path(country, window, experiment)
    if not (path.exists() and VALIDATION_FILE.exists()):
        return False
    saved = pd.read_csv(VALIDATION_FILE)
    family = BASELINE_EXPERIMENTS[experiment]
    recorded = ((saved["country"] == country) & (saved["window"] == window) & (saved["family"] == family)).any()
    if not recorded or joblib.load(path)["features"] != selected_features:
        raise ValueError(f"{path.name} exists but is incomplete or outdated. Move it before rerunning.")
    return True


def run_baseline(experiment, country, window, train, valid):
    """Search, save and record one Elastic Net or XGBoost configuration."""
    family = BASELINE_EXPERIMENTS[experiment]
    print(f"\n{family} | {country} | {window} | cpu")
    print(f"  Training: {train['info']['eom'].nunique()} months, {train['info']['id'].nunique():,} stocks, "
          f"{len(train['y']):,} labeled rows | Validation: {valid['info']['eom'].nunique()} months | "
          f"{len(selected_features)} features")
    model, trials = search(family, train, valid)
    trials = trials.assign(
        country=country, window=window, family=family, train_rows=len(train["y"]), validation_rows=len(valid["y"]),
        predictors=len(selected_features), settings=len(trials),
        zero_mse=monthly_mse(valid["info"], valid["y"], 0.0),
    )
    if VALIDATION_FILE.exists():
        saved = pd.read_csv(VALIDATION_FILE)
        same = (saved["country"] == country) & (saved["window"] == window) & (saved["family"] == family)
        trials = pd.concat([saved[~same], trials], ignore_index=True)
    save_safely(VALIDATION_FILE, lambda path: trials.to_csv(path, index=False))  # results first, model last
    saved_model = {"model": model, "features": selected_features}
    save_safely(baseline_path(country, window, experiment), lambda path: joblib.dump(saved_model, path))

In [257]:
for experiment in BASELINE_EXPERIMENTS:
    if not should_run(experiment):
        continue
    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    for country in LOCAL_COUNTRIES:
        valid = load_labeled(country, "validation")
        for window in TRAIN_WINDOWS[country]:
            if baseline_complete(country, window, experiment):
                print(f"{experiment} | {country} | {window}: skipped (saved model and results are complete)")
                continue
            train = load_labeled(country, window)
            assert train["info"]["predicted_month"].max() < valid["info"]["predicted_month"].min()
            run_baseline(experiment, country, window, train, valid)

elastic_net: not run (RUN_MODE = 'off')
xgboost: not run (RUN_MODE = 'off')


In [258]:
if VALIDATION_FILE.exists():
    validation_results = pd.read_csv(VALIDATION_FILE)
    best = validation_results[validation_results["selected"]]
    hyperparameters = ["alpha", "l1_ratio", "max_depth", "learning_rate", "n_estimators"]
    comparison = best[["country", "window", "family", "train_rows", "validation_rows", "predictors", "settings"]].assign(
        best_settings=[
            ", ".join(f"{name}={row[name]:g}" for name in hyperparameters if pd.notna(row[name]))
            for _, row in best.iterrows()
        ],
        validation_mse=best["validation_mse"], zero_mse=best["zero_mse"],
        max_abs_prediction=best["max_abs_prediction"], prediction_std=best["prediction_std"],
        converged=best["converged"],
    )
    display(comparison)

    not_converged = (~validation_results["converged"]).sum()
    print(f"Settings with convergence warnings: {not_converged} of {len(validation_results)}")
else:
    print("No validation results yet: the baseline experiments have not been run.")

No validation results yet: the baseline experiments have not been run.


## 16. Baseline Model Verification

The checks use the smoke-test fits, the configuration and, when available, the saved full-training results.

In [259]:
assert all(0 < len(grid) <= MAX_SETTINGS for _, grid in MODEL_FAMILIES.values())

# Every supervised period is complete, precedes the test period, and training precedes validation
for country in LOCAL_COUNTRIES:
    assert TRAIN_WINDOWS[country] == ["train_1y", "train_3y", "train_5y"]
    for window in TRAIN_WINDOWS[country]:
        assert PERIODS[window][1] < PERIODS["validation"][0] <= PERIODS["validation"][1] < PERIODS["test"][0]

if VALIDATION_FILE.exists():
    configuration = ["country", "window", "family"]
    assert (validation_results.groupby(configuration).size() <= MAX_SETTINGS).all()
    assert (validation_results.groupby(configuration)["selected"].sum() == 1).all()
    assert (validation_results["predictors"] == len(selected_features)).all()
print("Configuration and search limits verified.")

Configuration and search limits verified.


In [260]:
if RUN_SMOKE_TESTS:
    # Correct periods: no validation month in training, no test month anywhere
    assert in_period(smoke_train["info"]["predicted_month"], "train_1y").all()
    assert in_period(smoke_valid["info"]["predicted_month"], "validation").all()

    # Only labeled, finite targets, and equal total weight per month
    assert smoke_train["info"]["has_label"].all() and np.isfinite(smoke_train["y"]).all()
    weights = pd.Series(month_weights(smoke_train["info"]))
    month_totals = weights.groupby(smoke_train["info"]["eom"]).sum()
    assert np.allclose(month_totals, month_totals.iloc[0]) and np.isclose(weights.mean(), 1)

    for family, model in smoke_models.items():
        assert model.n_features_in_ == len(selected_features) == smoke_train["X"].shape[1]
        predictions = predict_model(model, smoke_valid)
        assert np.isfinite(predictions["prediction"]).all()

        # Each prediction is matched to the same stock and month in the source data
        matched = predictions.merge(data["GBR"][["id", "eom", TARGET]], on=["id", "eom"], validate="one_to_one")
        assert len(matched) == len(predictions) and np.array_equal(matched[TARGET], smoke_valid["y"])
    print("Periods, labels, weights, predictions and stock alignment verified.")

In [261]:
if RUN_SMOKE_TESTS:
    # Weights reach the models: putting all weight on one month must change the fit
    info = smoke_train["info"]
    one_month = (info["eom"] == info["eom"].max()).to_numpy()
    for family, model in smoke_models.items():
        make_model = MODEL_FAMILIES[family][0]
        params = {name: model.get_params()[name] for name in MODEL_FAMILIES[family][1][0]}
        weighted = make_model(params).fit(smoke_train["X"], smoke_train["y"], sample_weight=one_month.astype(float))
        unweighted = make_model(params).fit(smoke_train["X"], smoke_train["y"])
        assert not np.allclose(weighted.predict(smoke_valid["X"]), unweighted.predict(smoke_valid["X"]))
    print("Both models respond to sample weights.")

Phase 3 — End

Phase 4 — Start

## 17. Cross-Sectional Transformer Architecture

The Transformer receives all stocks of one country-month as one sequence and predicts each stock's next-month excess
return. Attention lets a prediction depend on the characteristics of other stocks in the same month. There are no
positional embeddings or stock identifiers, so the order of stocks carries no meaning.

In [262]:
def select_device(choice):
    """Use the requested device, or the first available of cuda, mps and cpu."""
    available = {"cuda": torch.cuda.is_available(), "mps": torch.backends.mps.is_available(), "cpu": True}
    if choice == "auto":
        choice = next(name for name, usable in available.items() if usable)
    if not available.get(choice, False):
        raise ValueError(f"Device '{choice}' is not available on this computer.")
    return torch.device(choice)


DEVICE = select_device(DEVICE_CHOICE)
print(f"Device: {DEVICE}")

N_HEADS = 4
FEEDFORWARD_DIM = 128
MAX_EPOCHS = RESEARCH_EPOCHS  # the proposal maximum is 100; LOCAL_RUN sets its own limit
PATIENCE = 10  # epochs without validation improvement before stopping
TRANSFORMER_SEEDS = [SEED, SEED + 1, SEED + 2]  # the first seed is also used for the hyperparameter search

DEFAULT_CONFIG = {
    "embedding_dim": 64, "n_blocks": 2, "dropout": 0.10, "learning_rate": 3e-4, "weight_decay": 1e-4,
}

Device: mps


In [263]:
class TransformerBlock(nn.Module):
    """Attention across stocks followed by a feed-forward layer, each with a residual connection."""

    def __init__(self, embedding_dim, dropout):
        super().__init__()
        self.dropout_rate = dropout
        self.attention_norm = nn.LayerNorm(embedding_dim)
        self.query_key_value = nn.Linear(embedding_dim, 3 * embedding_dim)
        self.attention_output = nn.Linear(embedding_dim, embedding_dim)
        self.feedforward_norm = nn.LayerNorm(embedding_dim)
        self.feedforward = nn.Sequential(
            nn.Linear(embedding_dim, FEEDFORWARD_DIM), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(FEEDFORWARD_DIM, embedding_dim),
        )
        self.dropout = nn.Dropout(dropout)

    def attend(self, x):
        n_stocks, embedding_dim = x.shape
        # Shape of query, key and value: (1 month, heads, stocks, dimension per head)
        query, key, value = [
            part.view(1, n_stocks, N_HEADS, embedding_dim // N_HEADS).transpose(1, 2)
            for part in self.query_key_value(x).chunk(3, dim=-1)
        ]
        attended = F.scaled_dot_product_attention(
            query, key, value, dropout_p=self.dropout_rate if self.training else 0.0
        )
        return self.attention_output(attended.transpose(1, 2).reshape(n_stocks, embedding_dim))

    def forward(self, x):
        x = x + self.dropout(self.attend(self.attention_norm(x)))
        return x + self.dropout(self.feedforward(self.feedforward_norm(x)))

In [264]:
class CrossSectionalTransformer(nn.Module):
    """Predict one return per stock from a (stocks, features) matrix of a single country-month."""

    def __init__(self, n_features, embedding_dim, n_blocks, dropout):
        super().__init__()
        self.input_layer = nn.Linear(n_features, embedding_dim)
        self.input_dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([TransformerBlock(embedding_dim, dropout) for _ in range(n_blocks)])
        self.output_norm = nn.LayerNorm(embedding_dim)
        self.output_layer = nn.Linear(embedding_dim, 1)

    def forward(self, x):
        x = self.input_dropout(self.input_layer(x))
        for block in self.blocks:
            x = block(x)
        return self.output_layer(self.output_norm(x)).squeeze(-1)


def make_transformer(config):
    """Build a new model on the selected device."""
    return CrossSectionalTransformer(
        len(selected_features), config["embedding_dim"], config["n_blocks"], config["dropout"]
    ).to(DEVICE)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## 18. Monthly Training and Validation

Each country-month is one training step, so every month has equal importance whatever its number of stocks. The order
of training months is shuffled each epoch with a fixed seed; stocks and months are never mixed. All stocks enter the
attention, but losses use labeled stocks only. Months without labels are skipped.

In [265]:
def make_months(info, X):
    """Split prepared data into one set of CPU tensors per country-month."""
    months = []
    for eom in info["eom"].unique():
        month_info, month_X = get_cross_section(info, X, eom)
        months.append({
            "info": month_info.reset_index(drop=True),
            "X": torch.from_numpy(month_X),
            "y": torch.tensor(month_info[TARGET].to_numpy(), dtype=torch.float32),
            "labeled": torch.tensor(month_info["has_label"].to_numpy(), dtype=torch.bool),
        })
    return months


def load_months(country, period):
    """Load training or validation months for one country."""
    if period == "test":
        raise ValueError("Test data cannot be used for training or model selection.")
    return make_months(*prepare_data(country, period, selected_features))


def month_mse(model, month):
    """Squared error averaged over the labeled stocks of one month."""
    labeled = month["labeled"].to(DEVICE)
    prediction = model(month["X"].to(DEVICE))
    return ((prediction[labeled] - month["y"].to(DEVICE)[labeled]) ** 2).mean()


def describe_month(month):
    first = month["info"].iloc[0]
    return f"{first['country']} {first['eom']:%Y-%m} ({len(month['info'])} stocks)"


def device_error(month, error):
    """Explain out-of-memory and unsupported-operation errors; other errors are returned unchanged."""
    if isinstance(error, torch.cuda.OutOfMemoryError) or "out of memory" in str(error).lower():
        return RuntimeError(
            f"{DEVICE} memory is too small for full attention on {describe_month(month)}. "
            "Use a device with more memory, or DEVICE_CHOICE = 'cpu'; the cross-section must not be split."
        )
    if isinstance(error, NotImplementedError):
        return RuntimeError(f"An operation is not supported on {DEVICE}. Set DEVICE_CHOICE = 'cpu'.")
    return error


In [266]:
def validate_transformer(model, months):
    """Monthly-equal MSE without gradients."""
    model.eval()
    losses = []
    with torch.inference_mode():
        for month in months:
            if month["labeled"].any():
                try:
                    losses.append(month_mse(model, month).item())
                except RuntimeError as error:
                    raise device_error(month, error) from error
    if not losses:
        raise ValueError("No labeled months are available for validation.")
    return float(np.mean(losses))


def predict_transformer(model, months):
    """Predictions for all stocks, aligned with identifiers and months."""
    model.eval()
    predictions = []
    with torch.inference_mode():
        for month in months:
            month_predictions = month["info"][["country", "id", "eom", "predicted_month"]].copy()
            try:
                month_predictions["prediction"] = model(month["X"].to(DEVICE)).cpu().numpy()
            except RuntimeError as error:
                raise device_error(month, error) from error
            predictions.append(month_predictions)
    return pd.concat(predictions, ignore_index=True)

In [267]:
def train_one_epoch(model, optimizer, months, order):
    """One optimizer step per month; returns the average monthly training MSE."""
    model.train()
    losses, last_update = [], time.time()
    for step, i in enumerate(order, start=1):
        month = months[i]
        optimizer.zero_grad(set_to_none=True)
        try:
            loss = month_mse(model, month)
            loss.backward()
        except RuntimeError as error:
            raise device_error(month, error) from error

        if not torch.isfinite(loss):
            raise FloatingPointError(f"Nonfinite loss in {describe_month(month)}.")
        if not all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None):
            raise FloatingPointError(f"Nonfinite gradient in {describe_month(month)}.")
        optimizer.step()
        losses.append(loss.item())
        if time.time() - last_update > 30:  # progress within long epochs
            print(f"    month {step}/{len(order)}, average training MSE {np.mean(losses):.6f}", flush=True)
            last_update = time.time()
    return float(np.mean(losses))

In [268]:
def print_run(label, train_months, valid_months, config, seed, max_epochs):
    """Describe a training run before it starts."""
    info = pd.concat([month["info"][["country", "id", "predicted_month"]] for month in train_months])
    labeled = sum(int(month["labeled"].sum()) for month in train_months)
    print(f"\nTransformer | {label} | seed {seed} | {DEVICE}")
    print(f"  Countries: {', '.join(info['country'].unique())} | Training: {len(train_months)} months, "
          f"{info['predicted_month'].min():%Y-%m} to {info['predicted_month'].max():%Y-%m}, "
          f"{info[['country', 'id']].drop_duplicates().shape[0]:,} stocks, {len(info):,} rows ({labeled:,} labeled)")
    print(f"  Validation: {len(valid_months)} months | Features: {len(selected_features)} | Maximum epochs: {max_epochs}")
    print("  Settings: " + ", ".join(f"{name}={value:g}" for name, value in config.items()), flush=True)

Training stops after `PATIENCE` epochs without a lower validation MSE, or after 100 epochs, and the best state is
restored. `initial_state` allows training to continue from existing weights.

In [269]:
def train_transformer(train_months, valid_months, config, seed, label, initial_state=None,
                      max_epochs=MAX_EPOCHS, patience=PATIENCE):
    """Train with early stopping; returns the best model and a record of the run."""
    if not 1 <= max_epochs <= 100:
        raise ValueError("max_epochs must be between 1 and 100.")
    torch.manual_seed(seed)  # also seeds CUDA and MPS
    shuffler = np.random.default_rng(seed)
    model = make_transformer(config)
    if initial_state is not None:
        model.load_state_dict(initial_state)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"]
    )
    train_months = [month for month in train_months if month["labeled"].any()]
    if not train_months:
        raise ValueError(f"{label}: no labeled training months.")

    print_run(label, train_months, valid_months, config, seed, max_epochs)

    started = time.time()
    best_mse, best_epoch, best_state, history = np.inf, 0, None, []
    for epoch in range(1, max_epochs + 1):
        train_mse = train_one_epoch(model, optimizer, train_months, shuffler.permutation(len(train_months)))
        valid_mse = validate_transformer(model, valid_months)
        if not np.isfinite(valid_mse):
            raise FloatingPointError(f"{label}: nonfinite validation MSE at epoch {epoch}.")
        history.append({"epoch": epoch, "train_mse": train_mse, "validation_mse": valid_mse})

        if valid_mse < best_mse:
            best_mse, best_epoch = valid_mse, epoch
            best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}

        stop = epoch - best_epoch >= patience
        print(f"  epoch {epoch}/{max_epochs} | train MSE {train_mse:.6f} | validation MSE {valid_mse:.6f} | "
              f"best {best_mse:.6f} | {time.time() - started:.0f}s", flush=True)
        if stop:
            print(f"  stopped early: no improvement for {patience} epochs", flush=True)
            break

    model.load_state_dict(best_state)
    run = {
        "config": dict(config), "seed": seed, "best_epoch": best_epoch, "validation_mse": best_mse,
        "n_parameters": count_parameters(model), "history": history, "max_epochs": max_epochs, "state": best_state,
    }
    return model, run

In [ ]:
if RUN_PREFLIGHT:
    # One training step of the largest model on the largest cross-section, with artificial inputs
    sizes = pd.Series({country: df.groupby("eom").size().max() for country, df in data.items()})
    print(f"Device: {DEVICE} | features: {len(selected_features)} | largest cross-sections: {sizes.to_dict()}")
    n_stocks = int(sizes.max())
    generator = np.random.default_rng(SEED)
    largest_month = {
        "info": pd.DataFrame({"country": sizes.idxmax(), "id": np.arange(n_stocks), "eom": pd.Timestamp("2000-01-31")}),
        "X": torch.from_numpy(generator.uniform(-0.5, 0.5, (n_stocks, len(selected_features))).astype("float32")),
        "y": torch.zeros(n_stocks), "labeled": torch.ones(n_stocks, dtype=torch.bool),
    }
    largest_model = make_transformer({**DEFAULT_CONFIG, "embedding_dim": 128, "n_blocks": 2})
    started = time.time()
    train_one_epoch(largest_model, torch.optim.AdamW(largest_model.parameters()), [largest_month], [0])
    print(f"Largest training step: {n_stocks:,} stocks in {time.time() - started:.1f} seconds")
    if DEVICE.type == "cuda":
        print(f"GPU: {torch.cuda.get_device_name()} | peak memory {torch.cuda.max_memory_allocated() / 1e9:.1f} GB of "
              f"{torch.cuda.get_device_properties(DEVICE).total_memory / 1e9:.1f} GB")
    del largest_model, largest_month

## 19. Transformer Hyperparameter Selection

The 18 settings cover all eight architectures (embedding 64/128, 1/2 blocks, dropout 10%/20%) at two learning rates,
plus a stronger weight decay for the default architecture at both learning rates. Selection uses only the 2018–2019
validation MSE.

In [270]:
TRANSFORMER_GRID = [
    {"embedding_dim": dim, "n_blocks": blocks, "dropout": dropout, "learning_rate": rate, "weight_decay": 1e-4}
    for dim, blocks, dropout, rate in product([64, 128], [2, 1], [0.10, 0.20], [3e-4, 1e-3])
]
TRANSFORMER_GRID += [{**DEFAULT_CONFIG, "learning_rate": rate, "weight_decay": 1e-2} for rate in [3e-4, 1e-3]]

print(f"Transformer settings: {len(TRANSFORMER_GRID)}")

Transformer settings: 18


In [271]:
def search_transformer(train_months, valid_months, label, grid=TRANSFORMER_GRID, initial_state=None):
    """Evaluate every setting with one seed; returns the best run and a table of all settings."""
    best_run, trials = None, []
    for number, config in enumerate(grid, start=1):
        trial_label = f"{label} | setting {number}/{len(grid)}"
        trial = {**config, "seed": SEED, "validation_mse": np.nan, "best_epoch": np.nan, "failure": ""}
        try:
            _, run = train_transformer(train_months, valid_months, config, SEED, trial_label, initial_state)
            trial.update(
                validation_mse=run["validation_mse"], best_epoch=run["best_epoch"], n_parameters=run["n_parameters"]
            )
            if best_run is None or run["validation_mse"] < best_run["validation_mse"]:
                best_run = run
        except FloatingPointError as error:  # unstable setting: record it and continue
            trial["failure"] = str(error)
            print(f"FAILED {trial_label}: {error}", flush=True)
        trials.append(trial)

    if best_run is None:
        raise RuntimeError(f"{label}: every setting became numerically unstable.")
    return best_run, pd.DataFrame(trials)


def train_seeds(train_months, valid_months, best_run, label, initial_state=None):
    """Train the selected setting from three seeds; the search run already provides the first."""
    runs = [best_run]
    for seed in TRANSFORMER_SEEDS:
        if seed != best_run["seed"]:
            _, run = train_transformer(train_months, valid_months, best_run["config"], seed, label, initial_state)
            runs.append(run)
    return runs

## 20. Local Transformer Experiments

`local_train` mode trains the default Transformer once on real data, using the country and history in `LOCAL_RUN`. Its
checkpoint is stored in the `development/` results folder and is never used as a research result.

`full_research` mode applies the complete search and three seeds to the UK and Japan with 1, 3 and 5 years of local data,
all ending in December 2017. Only the three selected runs per configuration are saved, each with everything needed to
rebuild the model. Configurations verified as complete are skipped, so interrupted training can be resumed.

In [272]:
TRANSFORMER_FILE = RESULTS_DIR / "transformer_local_validation.csv"
DEV_DIR = RESULTS_DIR / "development"


def checkpoint_path(experiment, country, window, seed):
    if experiment == "development":
        return DEV_DIR / f"DEV_{country}_{window}_transformer_seed{seed}.pt"
    return MODELS_DIR / f"{country}_{window}_transformer_{experiment}_seed{seed}.pt"


def save_checkpoint(run, experiment, country, window):
    checkpoint = {
        **run, "experiment": experiment, "country": country, "window": window, "n_heads": N_HEADS,
        "feedforward_dim": FEEDFORWARD_DIM, "n_features": len(selected_features), "features": list(selected_features),
    }
    save_safely(checkpoint_path(experiment, country, window, run["seed"]), lambda path: torch.save(checkpoint, path))


def load_checkpoint(path):
    """Rebuild a saved model; returns (model, checkpoint)."""
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    if checkpoint["features"] != selected_features:
        raise ValueError(f"{path.name}: saved predictors differ from selected_features.")
    if (checkpoint["n_heads"], checkpoint["feedforward_dim"]) != (N_HEADS, FEEDFORWARD_DIM):
        raise ValueError(f"{path.name}: saved architecture differs from the notebook settings.")
    model = make_transformer(checkpoint["config"])
    model.load_state_dict(checkpoint["state"])
    return model.eval(), checkpoint

In [273]:
def check_training_periods(train_months, valid_months, window):
    """Confirm that months belong to the permitted training and validation periods."""
    train_dates = pd.concat([month["info"]["predicted_month"] for month in train_months])
    valid_dates = pd.concat([month["info"]["predicted_month"] for month in valid_months])
    assert in_period(train_dates, window).all() and in_period(valid_dates, "validation").all()
    assert train_dates.max() < valid_dates.min() and valid_dates.max() < PERIODS["test"][0]


def run_local_experiment(country, window, valid_months):
    """Search and train three seeds for one country and history; returns the results table and the runs."""
    label = f"local | {country} | {window}"
    train_months = load_months(country, window)
    check_training_periods(train_months, valid_months, window)

    best_run, trials = search_transformer(train_months, valid_months, label)
    runs = train_seeds(train_months, valid_months, best_run, label)
    seed_rows = []
    for run in runs:
        seed_rows.append({
            **run["config"], "seed": run["seed"], "validation_mse": run["validation_mse"],
            "best_epoch": run["best_epoch"], "n_parameters": run["n_parameters"], "failure": "",
        })
    results = pd.concat([trials.assign(stage="search"), pd.DataFrame(seed_rows).assign(stage="seed")])
    return results.assign(country=country, window=window), runs

In [274]:
def all_seeds_saved(experiment, country, window):
    """True if all three seeds are saved as complete research runs."""
    paths = [checkpoint_path(experiment, country, window, seed) for seed in TRANSFORMER_SEEDS]
    if not all(path.exists() for path in paths):
        return False
    for seed, path in zip(TRANSFORMER_SEEDS, paths):
        _, checkpoint = load_checkpoint(path)
        identity = (checkpoint["experiment"], checkpoint["country"], checkpoint["window"], checkpoint["seed"])
        if identity != (experiment, country, window, seed) or checkpoint["max_epochs"] != MAX_EPOCHS:
            raise ValueError(f"{path.name} is not a complete research run. Move it before rerunning.")
    return True

In [275]:
if RUN_MODE == "local_train":
    country, window, seed = LOCAL_RUN["country"], LOCAL_RUN["window"], LOCAL_RUN["seed"]
    if window not in TRAIN_WINDOWS.get(country, []):
        raise ValueError(f"LOCAL_RUN: choose a window from {TRAIN_WINDOWS} (country first).")

    print("Loading real data for the development run...")
    dev_train = load_months(country, window)
    dev_valid = load_months(country, "validation")
    check_training_periods(dev_train, dev_valid, window)
    dev_model, dev_run = train_transformer(
        dev_train, dev_valid, DEFAULT_CONFIG, seed, f"DEVELOPMENT | {country} | {window}",
        max_epochs=LOCAL_RUN["max_epochs"],
    )

    DEV_DIR.mkdir(parents=True, exist_ok=True)
    save_checkpoint(dev_run, "development", country, window)
    print(f"\nBest epoch {dev_run['best_epoch']}, validation MSE {dev_run['validation_mse']:.6f}, "
          f"{dev_run['n_parameters']:,} parameters")
    print(f"Development checkpoint: {checkpoint_path('development', country, window, seed)}")
    print("This is a limited one-seed run, not a research result.")
else:
    print(f"Development run: not run (RUN_MODE = '{RUN_MODE}')")

Development run: not run (RUN_MODE = 'off')


In [276]:
if RUN_MODE == "local_train":
    # Compare with forecasting zero, and reload the saved checkpoint
    zero_mse = np.mean([(m["y"][m["labeled"]] ** 2).mean().item() for m in dev_valid if m["labeled"].any()])
    reloaded, _ = load_checkpoint(checkpoint_path("development", country, window, seed))
    assert np.isclose(validate_transformer(reloaded, dev_valid), dev_run["validation_mse"], rtol=1e-4)

    display(pd.DataFrame(dev_run["history"]).set_index("epoch"))
    print(f"Zero-forecast validation MSE: {zero_mse:.6f}")
    display(predict_transformer(reloaded, dev_valid[:1]).head())

In [277]:
if should_run("local"):
    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    for country in LOCAL_COUNTRIES:
        valid_months = load_months(country, "validation")
        for window in TRAIN_WINDOWS[country]:
            saved = pd.DataFrame(columns=["country", "window"])
            if TRANSFORMER_FILE.exists():
                saved = pd.read_csv(TRANSFORMER_FILE)
            this_run = (saved["country"] == country) & (saved["window"] == window)
            if all_seeds_saved("local", country, window):
                if not this_run.any():
                    raise ValueError(f"local | {country} | {window}: models exist but results are missing. Move them first.")
                print(f"local | {country} | {window}: skipped (three verified checkpoints and results exist)")
                continue

            results, runs = run_local_experiment(country, window, valid_months)
            table = pd.concat([saved[~this_run], results], ignore_index=True)
            save_safely(TRANSFORMER_FILE, lambda path: table.to_csv(path, index=False))  # results first, models last
            for run in runs:
                save_checkpoint(run, "local", country, window)

local: not run (RUN_MODE = 'off')


In [278]:
if TRANSFORMER_FILE.exists():
    transformer_results = pd.read_csv(TRANSFORMER_FILE)
    seed_runs = transformer_results[transformer_results["stage"] == "seed"]
    display(seed_runs[[
        "country", "window", "seed", *DEFAULT_CONFIG, "best_epoch", "validation_mse", "n_parameters"
    ]])
else:
    print("No Transformer results yet: the local experiments have not been run.")

No Transformer results yet: the local experiments have not been run.


## 21. Transformer Verification

Configuration checks always run. The smoke test uses small artificial cross-sections and saves nothing.

In [279]:
assert len(TRANSFORMER_GRID) <= MAX_SETTINGS and DEFAULT_CONFIG in TRANSFORMER_GRID
assert all(TRANSFORMER_GRID.count(config) == 1 for config in TRANSFORMER_GRID)
assert all(config["embedding_dim"] % N_HEADS == 0 for config in TRANSFORMER_GRID)
assert len(set(TRANSFORMER_SEEDS)) == 3 and SEED in TRANSFORMER_SEEDS
assert 1 <= MAX_EPOCHS <= 100

for country in LOCAL_COUNTRIES:
    for window in TRAIN_WINDOWS[country]:
        assert PERIODS[window][1] < PERIODS["validation"][0] <= PERIODS["validation"][1] < PERIODS["test"][0]
print("Transformer configuration verified.")

Transformer configuration verified.


In [280]:
def make_smoke_months(sizes, first_eom):
    """Artificial cross-sections with the same structure as prepared data; two stocks per month lack labels."""
    generator = np.random.default_rng(SEED)
    info = pd.concat([
        pd.DataFrame({"country": "GBR", "id": np.arange(size), "eom": eom})
        for size, eom in zip(sizes, pd.date_range(first_eom, periods=len(sizes), freq="ME"))
    ], ignore_index=True)
    info["predicted_month"] = info["eom"] + pd.offsets.MonthEnd(1)
    info[TARGET] = generator.normal(0, 0.1, len(info))
    info.loc[info["id"] < 2, TARGET] = np.nan
    info["has_label"] = info[TARGET].notna()
    X = generator.uniform(-0.5, 0.5, (len(info), len(selected_features))).astype("float32")
    return info, X

In [281]:
if RUN_SMOKE_TESTS:
    print("Checking shapes, alignment, masking and gradients...")
    smoke_info, smoke_X = make_smoke_months([20, 35, 27], "2017-09-30")
    smoke_train_months = make_months(smoke_info, smoke_X)
    smoke_valid_months = make_months(*make_smoke_months([25, 30], "2017-12-31"))
    check_training_periods(smoke_train_months, smoke_valid_months, "train_1y")

    torch.manual_seed(SEED)
    model = make_transformer(DEFAULT_CONFIG).eval()
    month = smoke_train_months[1]
    assert model.input_layer.in_features == len(selected_features)

    # Variable stock counts, finite outputs and identifier alignment
    predictions = predict_transformer(model, smoke_train_months)
    assert np.isfinite(predictions["prediction"]).all()
    keys = ["country", "id", "eom", "predicted_month"]
    assert predictions[keys].equals(smoke_info[keys].reset_index(drop=True))

    # Reordering stocks reorders predictions in the same way
    order = torch.randperm(len(month["X"]))
    with torch.inference_mode():
        original = model(month["X"].to(DEVICE)).cpu()
        reordered = model(month["X"][order].to(DEVICE)).cpu()
    assert original.shape == (len(month["X"]),)
    assert torch.allclose(original[order], reordered, atol=1e-5)

    # Predictions depend on the other stocks in the month
    with torch.inference_mode():
        alone = model(month["X"][:1].to(DEVICE)).cpu()
    assert not torch.allclose(alone, original[:1])

In [282]:
if RUN_SMOKE_TESTS:
    # Unlabeled targets never affect the loss, and months are averaged equally
    changed = {**month, "y": torch.where(month["labeled"], month["y"], torch.tensor(1e6))}
    with torch.inference_mode():
        assert torch.equal(month_mse(model, month), month_mse(model, changed))
        monthly_losses = [month_mse(model, m).item() for m in smoke_train_months]
    assert np.isclose(validate_transformer(model, smoke_train_months), np.mean(monthly_losses))

    # A month without labels is skipped
    unlabeled = {**month, "labeled": torch.zeros_like(month["labeled"])}
    assert validate_transformer(model, [month, unlabeled]) == validate_transformer(model, [month])

    # Finite gradients and a real parameter update
    before = [p.detach().clone() for p in model.parameters()]
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    train_one_epoch(model, optimizer, smoke_train_months, range(len(smoke_train_months)))
    assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters())
    assert any(not torch.equal(old, new) for old, new in zip(before, model.parameters()))
    print("Loss masking, monthly averaging, gradients and updates verified.")

In [283]:
if RUN_SMOKE_TESTS:
    # Same seed gives the same run; a different seed gives a different model
    _, first = train_transformer(smoke_train_months, smoke_valid_months, DEFAULT_CONFIG, SEED, "Smoke", max_epochs=3)
    _, second = train_transformer(smoke_train_months, smoke_valid_months, DEFAULT_CONFIG, SEED, "Smoke", max_epochs=3)
    _, other = train_transformer(smoke_train_months, smoke_valid_months, DEFAULT_CONFIG, SEED + 1, "Smoke", max_epochs=3)
    assert np.isclose(first["validation_mse"], second["validation_mse"], rtol=1e-4)
    assert any(not torch.equal(first["state"][name], other["state"][name]) for name in first["state"])

    # The returned model is the best validation state
    best_model = make_transformer(DEFAULT_CONFIG)
    best_model.load_state_dict(first["state"])
    assert np.isclose(validate_transformer(best_model, smoke_valid_months), first["validation_mse"])

    # With no learning, validation never improves, so training stops after one epoch plus the patience
    frozen = {**DEFAULT_CONFIG, "learning_rate": 0.0}
    _, stopped = train_transformer(smoke_train_months, smoke_valid_months, frozen, SEED, "Smoke", patience=2)
    assert stopped["best_epoch"] == 1 and len(stopped["history"]) == 3
    print("Reproducibility, best-state restoration and early stopping verified.")

Phase 4 — End

Phase 5 — Start

## 22. U.S. Transformer Pretraining

The source model is trained on U.S. predicted returns from February 2000 to December 2017 and selected on U.S.
2018–2019 validation data only. Its three seeds are the starting point for direct transfer and fine-tuning.

In [284]:
TRANSFER_FILE = RESULTS_DIR / "transfer_validation.csv"
ARCHITECTURE = ["embedding_dim", "n_blocks", "dropout"]


def architecture(config):
    """The settings that must match between compared models."""
    return {name: config[name] for name in ARCHITECTURE}

In [285]:
def rows_of(results, experiment, country, window):
    """Rows of one experiment in the results table."""
    return (
        (results["experiment"] == experiment) & (results["country"] == country) & (results["window"] == window)
    )


def read_transfer_results():
    if TRANSFER_FILE.exists():
        return pd.read_csv(TRANSFER_FILE)
    return pd.DataFrame(columns=["experiment", "country", "window"])


def record_results(table, experiment, country, window):
    """Add one experiment to the results file, replacing earlier rows of the same experiment."""
    saved = read_transfer_results()
    saved = saved[~rows_of(saved, experiment, country, window)]
    table = table.assign(experiment=experiment, country=country, window=window)
    combined = pd.concat([saved, table], ignore_index=True)
    save_safely(TRANSFER_FILE, lambda path: combined.to_csv(path, index=False))


def is_complete(experiment, country, window, sources=None):
    """True if all seeds are saved and recorded; saved models must match the current U.S. source."""
    paths = [checkpoint_path(experiment, country, window, seed) for seed in TRANSFORMER_SEEDS]
    recorded = rows_of(read_transfer_results(), experiment, country, window).any()
    if not all_seeds_saved(experiment, country, window):
        return False
    if not recorded:
        raise ValueError(f"{paths[0].name} and its other seeds exist but are not in {TRANSFER_FILE}. Move them first.")

    for seed, path in zip(TRANSFORMER_SEEDS, paths):
        _, checkpoint = load_checkpoint(path)
        if sources is not None and architecture(checkpoint["config"]) != architecture(sources[seed]["config"]):
            raise ValueError(f"{path.name} has a different architecture from the U.S. model. Move it before rerunning.")
        if experiment == "finetuned" and checkpoint["source_mse"] != sources[seed]["validation_mse"]:
            raise ValueError(f"{path.name} came from a different U.S. checkpoint. Move it before rerunning.")
    print(f"{experiment} | {country} | {window}: skipped (three verified checkpoints and results exist)")
    return True

In [286]:
def train_experiment(experiment, country, window, train_months, valid_months, grid, sources=None):
    """Search, train three seeds, save and record one experiment.

    With `sources`, each seed starts from the U.S. checkpoint of the same seed; otherwise weights are random.
    """
    label = f"{experiment} | {country} | {window}"
    if sources is None:
        best_run, trials = search_transformer(train_months, valid_months, label, grid)
        runs = train_seeds(train_months, valid_months, best_run, label)
    else:
        best_run, trials = search_transformer(train_months, valid_months, label, grid, sources[SEED]["state"])
        runs = [best_run]
        for seed in TRANSFORMER_SEEDS:
            if seed != SEED:
                _, run = train_transformer(
                    train_months, valid_months, best_run["config"], seed, label, sources[seed]["state"]
                )
                runs.append(run)

    rows = []
    for run in runs:
        seed = run["seed"]
        source_name = "" if sources is None else checkpoint_path("pretrained", "USA", "train_full", seed).name
        run["source_checkpoint"] = source_name
        run["source_mse"] = float("nan") if sources is None else sources[seed]["validation_mse"]
        rows.append({
            **run["config"], "seed": seed, "best_epoch": run["best_epoch"], "validation_mse": run["validation_mse"],
            "n_parameters": run["n_parameters"], "checkpoint": checkpoint_path(experiment, country, window, seed).name,
            "source_checkpoint": source_name,
        })
    table = pd.concat([trials.assign(stage="search"), pd.DataFrame(rows).assign(stage="seed")])
    record_results(table, experiment, country, window)  # results first, models last
    for run in runs:
        save_checkpoint(run, experiment, country, window)

In [287]:
if should_run("pretrained"):
    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    if not is_complete("pretrained", "USA", "train_full"):
        us_valid = load_months("USA", "validation")
        us_train = load_months("USA", "train_full")
        check_training_periods(us_train, us_valid, "train_full")
        train_experiment("pretrained", "USA", "train_full", us_train, us_valid, TRANSFORMER_GRID)
        del us_train, us_valid

pretrained: not run (RUN_MODE = 'off')


In [288]:
def load_us_sources():
    """Load the three U.S. checkpoints, which must share one configuration."""
    sources = {}
    for seed in TRANSFORMER_SEEDS:
        path = checkpoint_path("pretrained", "USA", "train_full", seed)
        if not path.exists():
            raise FileNotFoundError(f"U.S. checkpoint {path} not found. Run U.S. pretraining first.")
        _, checkpoint = load_checkpoint(path)
        if (checkpoint["experiment"], checkpoint["country"], checkpoint["seed"]) != ("pretrained", "USA", seed):
            raise ValueError(f"{path.name} is not the expected U.S. pretrained model.")
        sources[seed] = checkpoint

    if any(source["config"] != sources[SEED]["config"] for source in sources.values()):
        raise ValueError("The U.S. checkpoints have different configurations.")
    return sources

## 23. Direct Cross-Market Transfer

Each U.S. model is applied to UK and Japanese cross-sections without any weight update. Direct transfer uses no local
training data, so one set of U.S. models serves all three local-history comparisons and nothing new is saved.

In [289]:
def predict_direct_transfer(months, seed):
    """Forecasts of the U.S. model for another market's cross-sections, with weights unchanged."""
    model, _ = load_checkpoint(checkpoint_path("pretrained", "USA", "train_full", seed))
    return predict_transformer(model, months)


def validate_direct_transfer(country):
    """Validation MSE of each U.S. model in another market; returns a results table."""
    valid_months = load_months(country, "validation")
    rows = []
    for seed, source in load_us_sources().items():
        path = checkpoint_path("pretrained", "USA", "train_full", seed)
        model, _ = load_checkpoint(path)
        mse = validate_transformer(model, valid_months)
        assert all(torch.equal(value.cpu(), source["state"][name]) for name, value in model.state_dict().items())
        rows.append({
            **source["config"], "seed": seed, "best_epoch": source["best_epoch"], "validation_mse": mse,
            "n_parameters": source["n_parameters"], "checkpoint": path.name, "source_checkpoint": path.name,
        })
    return pd.DataFrame(rows).assign(stage="seed")

In [290]:
if should_run("direct"):
    for country in LOCAL_COUNTRIES:
        print(f"Validating direct transfer: USA to {country}...")
        record_results(validate_direct_transfer(country), "direct", country, "all")

direct: not run (RUN_MODE = 'off')


## 24. Local Fine-Tuning

A model with the U.S. architecture starts from the U.S. weights of the same seed and continues training on local data.
All parameters are updated. Only the learning rate and weight decay are tuned, and every learning rate is smaller than
in U.S. pretraining. Weights are loaded strictly, so incompatible architectures raise an error.

In [291]:
def fine_tuning_grid(source_config):
    """Smaller learning rates and two weight decays, with the source architecture fixed."""
    return [
        {**source_config, "learning_rate": factor * source_config["learning_rate"], "weight_decay": decay}
        for factor, decay in product([0.3, 0.1, 0.03], [1e-4, 1e-2])
    ]


if should_run("finetuned"):
    sources = load_us_sources()
    grid = fine_tuning_grid(sources[SEED]["config"])
    for country in LOCAL_COUNTRIES:
        valid_months = load_months(country, "validation")
        for window in TRAIN_WINDOWS[country]:
            if is_complete("finetuned", country, window, sources):
                continue
            train_months = load_months(country, window)
            check_training_periods(train_months, valid_months, window)
            train_experiment("finetuned", country, window, train_months, valid_months, grid, sources)

finetuned: not run (RUN_MODE = 'off')


## 25. Joint U.S.–Local Training

A randomly initialized model is trained on U.S. (2000–2017) and local country-months together. Each country-month
remains its own cross-section, so U.S. and local stocks never attend to each other. Selection uses the destination
country's validation data.

Every country-month has equal weight. With 215 U.S. months, local months are about 5% (1 year), 14% (3 years) and 22%
(5 years) of the training objective; no extra weighting is applied.

For controlled comparisons, joint models use the U.S.-selected architecture, and only the learning rate and weight
decay are tuned.

In [292]:
def matched_grid(source_config):
    """Fixed source architecture with a small learning-rate and weight-decay search."""
    return [
        {**architecture(source_config), "learning_rate": rate, "weight_decay": decay}
        for rate, decay in product([1e-4, 3e-4, 1e-3], [1e-4, 1e-2])
    ]


def join_months(us_months, local_months):
    """Combine country-months without merging any cross-sections."""
    joined = us_months + local_months
    assert all(month["info"][["country", "eom"]].drop_duplicates().shape[0] == 1 for month in joined)
    return joined

In [293]:
if should_run("joint"):
    sources = load_us_sources()  # provides the architecture only, never the weights
    grid = matched_grid(sources[SEED]["config"])
    us_months = None
    for country in LOCAL_COUNTRIES:
        valid_months = load_months(country, "validation")
        for window in TRAIN_WINDOWS[country]:
            if is_complete("joint", country, window, sources):
                continue
            if us_months is None:  # load the U.S. history once, and only if needed
                us_months = load_months("USA", "train_full")
            local_months = load_months(country, window)
            check_training_periods(us_months, valid_months, "train_full")
            check_training_periods(local_months, valid_months, window)
            train_experiment("joint", country, window, join_months(us_months, local_months), valid_months, grid)
    del us_months

joint: not run (RUN_MODE = 'off')


## 26. Transfer Experiment Configuration

Section 20 tunes the local-only architecture freely, so it may differ from the U.S. architecture. The primary comparison
therefore uses matched local-only models: random initialization, local data only, the U.S. architecture and the same
search as joint training. They are saved separately and the Section 20 models are kept.

| Strategy | Experiment | Initial weights | Training data | Search |
|---|---|---|---|---|
| Local only, freely tuned | `local` | Random | Local | 18 settings |
| Local only, matched | `localmatched` | Random | Local | 6 settings |
| Direct transfer | `direct` | U.S. model | None | None |
| Fine-tuning | `finetuned` | U.S. model | Local | 6 settings |
| Joint training | `joint` | Random | U.S. + local | 6 settings |

The order of execution is U.S. pretraining, direct transfer, fine-tuning, joint training and matched local models.
Run one notebook at a time, because all experiments share one results file.

In [294]:
if should_run("localmatched"):
    sources = load_us_sources()  # provides the architecture only, never the weights
    grid = matched_grid(sources[SEED]["config"])
    for country in LOCAL_COUNTRIES:
        valid_months = load_months(country, "validation")
        for window in TRAIN_WINDOWS[country]:
            if is_complete("localmatched", country, window, sources):
                continue
            train_months = load_months(country, window)
            check_training_periods(train_months, valid_months, window)
            train_experiment("localmatched", country, window, train_months, valid_months, grid)

localmatched: not run (RUN_MODE = 'off')


In [295]:
if TRANSFER_FILE.exists():
    transfer_results = read_transfer_results()
    seed_runs = transfer_results[transfer_results["stage"] == "seed"]
    print("Validation results by seed:")
    display(seed_runs[[
        "experiment", "country", "window", "seed", *DEFAULT_CONFIG, "best_epoch", "validation_mse", "n_parameters",
        "checkpoint", "source_checkpoint",
    ]])

    # Compare the freely tuned local architectures with the U.S. architecture
    us_rows = seed_runs[rows_of(seed_runs, "pretrained", "USA", "train_full")]
    if TRANSFORMER_FILE.exists() and not us_rows.empty:
        local_runs = pd.read_csv(TRANSFORMER_FILE).query("stage == 'seed'")
        local_runs = local_runs.drop_duplicates(["country", "window"])[["country", "window", *ARCHITECTURE]]
        local_runs["matches_us"] = (local_runs[ARCHITECTURE] == us_rows[ARCHITECTURE].iloc[0]).all(axis=1)
        print("Freely tuned local-only architectures:")
        display(local_runs)
else:
    print("No transfer results yet: the transfer experiments have not been run.")

No transfer results yet: the transfer experiments have not been run.


## 27. Transfer Learning Verification

Configuration checks always run. Checkpoint checks apply to any saved models, and the smoke test uses the artificial
months of Section 21 when `RUN_SMOKE_TESTS` is on.

In [296]:
fine_grid, joint_grid = fine_tuning_grid(DEFAULT_CONFIG), matched_grid(DEFAULT_CONFIG)

assert max(len(TRANSFORMER_GRID), len(fine_grid), len(joint_grid)) <= MAX_SETTINGS
assert all(architecture(config) == architecture(DEFAULT_CONFIG) for config in fine_grid + joint_grid)
assert all(config["learning_rate"] < DEFAULT_CONFIG["learning_rate"] for config in fine_grid)
assert len(set(TRANSFORMER_SEEDS)) == 3 and SEED in TRANSFORMER_SEEDS

# U.S. and local training end before validation, which ends before testing
assert TRAIN_WINDOWS["USA"] == ["train_full"]
assert all(PERIODS[window][1] < PERIODS["validation"][0] for windows in TRAIN_WINDOWS.values() for window in windows)
assert PERIODS["validation"][1] < PERIODS["test"][0]
print("Transfer configuration verified.")

Transfer configuration verified.


In [297]:
EXPERIMENTS = [("pretrained", "USA", "train_full")] + [
    (experiment, country, window)
    for experiment in ["local", "localmatched", "finetuned", "joint"]
    for country in LOCAL_COUNTRIES for window in TRAIN_WINDOWS[country]
]
paths = [checkpoint_path(*names, seed) for names in EXPERIMENTS for seed in TRANSFORMER_SEEDS]
assert len(set(paths)) == len(paths)  # every model has its own file

n_checked = 0
for names in EXPERIMENTS:
    for seed in TRANSFORMER_SEEDS:
        path = checkpoint_path(*names, seed)
        if not path.exists():
            continue
        _, checkpoint = load_checkpoint(path)  # also checks the predictor order and strict weight loading
        assert (checkpoint["experiment"], checkpoint["country"], checkpoint["window"]) == names
        assert checkpoint["seed"] == seed
        if names[0] in ["localmatched", "finetuned", "joint"]:
            _, source = load_checkpoint(checkpoint_path("pretrained", "USA", "train_full", seed))
            assert architecture(checkpoint["config"]) == architecture(source["config"])
            if names[0] == "finetuned":
                assert checkpoint["source_mse"] == source["validation_mse"]
                assert checkpoint["config"]["learning_rate"] < source["config"]["learning_rate"]
            else:
                assert checkpoint["source_checkpoint"] == ""  # random initialization
        n_checked += 1
print(f"Saved Transformer checkpoints verified: {n_checked} of {len(paths)}")

Saved Transformer checkpoints verified: 0 of 75


In [298]:
if RUN_SMOKE_TESTS:
    us_info, us_X = make_smoke_months([40, 32], "2017-10-31")
    us_smoke = make_months(us_info.assign(country="USA"), us_X)
    local_smoke = make_months(*make_smoke_months([20, 25], "2017-10-31"))
    valid_smoke = make_months(*make_smoke_months([22, 28], "2017-12-31"))
    check_training_periods(us_smoke, valid_smoke, "train_full")
    check_training_periods(local_smoke, valid_smoke, "train_1y")

    model, source = train_transformer(us_smoke, us_smoke, DEFAULT_CONFIG, SEED, "Smoke source", max_epochs=2)

    # Direct transfer: predictions are aligned and weights do not change
    predictions = predict_transformer(model, local_smoke)
    local_keys = pd.concat([month["info"][["id", "eom"]] for month in local_smoke], ignore_index=True)
    assert predictions[["id", "eom"]].equals(local_keys)
    assert all(torch.equal(value.cpu(), source["state"][name]) for name, value in model.state_dict().items())

In [299]:
if RUN_SMOKE_TESTS:
    def same_weights(first, second):
        return all(torch.equal(first[name], second[name]) for name in first)

    # Fine-tuning starts exactly from the source weights (a zero learning rate leaves them unchanged), then updates them
    frozen = {**DEFAULT_CONFIG, "learning_rate": 0.0}
    _, start = train_transformer(
        local_smoke, valid_smoke, frozen, SEED, "Smoke start", source["state"], max_epochs=1
    )
    _, tuned = train_transformer(
        local_smoke, valid_smoke, fine_grid[0], SEED, "Smoke fine-tuning", source["state"], max_epochs=1
    )
    assert same_weights(start["state"], source["state"])
    assert not same_weights(tuned["state"], source["state"])

    # Joint training starts from random weights and keeps the two markets in separate cross-sections
    joint_smoke = join_months(us_smoke, local_smoke)
    assert len(joint_smoke) == len(us_smoke) + len(local_smoke)
    _, joint_start = train_transformer(joint_smoke, valid_smoke, frozen, SEED, "Smoke joint", max_epochs=1)
    assert not same_weights(joint_start["state"], source["state"])

    # Different architectures cannot be loaded silently
    try:
        make_transformer({**DEFAULT_CONFIG, "embedding_dim": 128}).load_state_dict(source["state"])
        mismatch_rejected = False
    except RuntimeError:
        mismatch_rejected = True
    assert mismatch_rejected
    print("Direct transfer, fine-tuning and joint-training behavior verified.")

Phase 5 — End

Phase 6 — Start

## 28. Model Checkpoint Readiness

Predictions use only models saved by earlier sections; nothing is trained here. The primary comparison has six strategies.
The independently tuned local-only Transformer (`local`) is predicted separately and is not part of the ensemble.

In [ ]:
BASELINE_STRATEGIES = ["elastic_net", "xgboost"]
PRIMARY_STRATEGIES = BASELINE_STRATEGIES + ["localmatched", "direct", "finetuned", "joint"]
ALL_STRATEGIES = PRIMARY_STRATEGIES + ["local"]
TRANSFORMER_STRATEGIES = [s for s in ALL_STRATEGIES if s not in BASELINE_STRATEGIES]


def us_checkpoint_path(seed):
    return checkpoint_path("pretrained", "USA", "train_full", seed)


def prediction_files(strategy, country, window):
    """Saved model file of each seed; direct transfer uses the same U.S. files for every country and window."""
    if strategy in BASELINE_STRATEGIES:
        return {"single": baseline_path(country, window, strategy)}
    if strategy == "direct":
        return {str(seed): us_checkpoint_path(seed) for seed in TRANSFORMER_SEEDS}
    return {str(seed): checkpoint_path(strategy, country, window, seed) for seed in TRANSFORMER_SEEDS}


def checkpoint_readiness():
    """List every required model file and whether it exists."""
    return pd.DataFrame([
        {"country": country, "window": window, "strategy": strategy, "seed": seed, "file": path.name,
         "available": path.exists()}
        for country in LOCAL_COUNTRIES for window in TRAIN_WINDOWS[country] for strategy in ALL_STRATEGIES
        for seed, path in prediction_files(strategy, country, window).items()
    ])

In [ ]:
def load_baseline(country, window, strategy):
    """Load a saved Elastic Net or XGBoost model and check its predictors."""
    path = baseline_path(country, window, strategy)
    saved = joblib.load(path)
    if saved["features"] != selected_features or saved["model"].n_features_in_ != len(selected_features):
        raise ValueError(f"{path.name}: saved predictors differ from selected_features.")
    return saved["model"]


def load_transformer(country, window, strategy, seed):
    """Load a saved Transformer and check its identity, training length and architecture."""
    if strategy == "direct":
        path, identity = us_checkpoint_path(seed), ("pretrained", "USA", "train_full", seed)
    else:
        path, identity = checkpoint_path(strategy, country, window, seed), (strategy, country, window, seed)

    model, checkpoint = load_checkpoint(path)  # checks predictor names and order
    if tuple(checkpoint[name] for name in ["experiment", "country", "window", "seed"]) != identity:
        raise ValueError(f"{path.name}: the checkpoint belongs to a different experiment.")
    if checkpoint["max_epochs"] != MAX_EPOCHS:
        raise ValueError(f"{path.name}: trained with a maximum of {checkpoint['max_epochs']} epochs, not {MAX_EPOCHS}.")

    if strategy in ["localmatched", "finetuned", "joint"]:
        _, source = load_checkpoint(us_checkpoint_path(seed))
        if architecture(checkpoint["config"]) != architecture(source["config"]):
            raise ValueError(f"{path.name}: the architecture differs from the U.S. model.")
        if strategy == "finetuned" and checkpoint["source_mse"] != source["validation_mse"]:
            raise ValueError(f"{path.name}: fine-tuned from a different U.S. checkpoint.")
    return model

In [ ]:
if RUN_PHASE6_PREDICTIONS:
    readiness = checkpoint_readiness()
    missing_models = readiness[~readiness["available"]]
    print(f"Model files available: {readiness['available'].sum()} of {len(readiness)}")
    if not missing_models.empty:
        display(missing_models)
        raise FileNotFoundError(f"{len(missing_models)} required model files are missing; see the table above.")
else:
    print("Checkpoint readiness: not checked (RUN_PHASE6_PREDICTIONS = False)")

## 29. Out-of-Sample Predictions

Test data are loaded by a separate inference-only function; the training loaders still reject the test period. Every
available stock receives a prediction, including those without an actual return. Transformers see complete monthly
cross-sections in evaluation mode, without gradients.

In [ ]:
KEYS = ["country", "id", "eom", "predicted_month"]


def load_test_for_inference(country):
    """Test-period data for inference only; never used for training or model selection."""
    info, X = prepare_data(country, "test", selected_features)
    assert in_period(info["predicted_month"], "test").all()
    assert info["predicted_month"].nunique() == expected_months["test"]
    return info, X


def predict_baseline(model, info, X):
    """Baseline predictions for every stock, with identifiers."""
    predictions = info[KEYS].copy()
    predictions["prediction"] = model.predict(X)
    return predictions


def check_predictions(predictions, info, label):
    """Require one finite prediction for every stock-month."""
    if predictions.duplicated(KEYS).any():
        raise ValueError(f"{label}: duplicate stock-months in the predictions.")
    if len(predictions) != len(info):
        raise ValueError(f"{label}: {len(predictions):,} predictions for {len(info):,} stock-months.")
    if not np.isfinite(predictions["prediction"]).all():
        raise ValueError(f"{label}: missing or infinite predictions.")

In [ ]:
def predict_strategy(country, window, strategy, info, X, months):
    """Predictions of one strategy for each of its saved models: {seed: predictions}."""
    results = {}
    for seed in prediction_files(strategy, country, window):
        if strategy in BASELINE_STRATEGIES:
            predictions = predict_baseline(load_baseline(country, window, strategy), info, X)
        else:
            predictions = predict_transformer(load_transformer(country, window, strategy, int(seed)), months)
        check_predictions(predictions, info, f"{country} {window} {strategy} {seed}")
        results[seed] = predictions
    return results

## 30. Ensemble Construction

The rule is fixed before any test outcome is examined. Transformer forecasts are first averaged over the three seeds.
The ensemble is then the equal-weight average of the six primary strategies within each country and training history,
so a strategy with three seeds does not receive more weight. A partial ensemble is never produced.

In [ ]:
ENSEMBLE_WEIGHTS = {strategy: 1 / len(PRIMARY_STRATEGIES) for strategy in PRIMARY_STRATEGIES}


def final_column(strategy):
    """Column holding the strategy's final forecast: one model, or the average of three seeds."""
    return f"{strategy}|single" if strategy in BASELINE_STRATEGIES else f"{strategy}|average"


def add_seed_averages(wide):
    """Average the three seeds of every Transformer strategy."""
    for strategy in TRANSFORMER_STRATEGIES:
        wide[f"{strategy}|average"] = wide[[f"{strategy}|{seed}" for seed in TRANSFORMER_SEEDS]].mean(axis=1, skipna=False)
    return wide


def add_ensemble(wide):
    """Equal-weight average of the primary strategies; all of them must be present and complete."""
    columns = {strategy: final_column(strategy) for strategy in PRIMARY_STRATEGIES}
    absent = [strategy for strategy, column in columns.items() if column not in wide or wide[column].isna().any()]
    if absent:
        raise ValueError(f"The ensemble cannot be built; incomplete strategies: {absent}")
    wide["ensemble|ensemble"] = sum(ENSEMBLE_WEIGHTS[strategy] * wide[column] for strategy, column in columns.items())
    return wide

## 31. Prediction Alignment and Storage

Forecasts are joined by country, stock and month, never by row position. `in_common` marks stock-months predicted by all
primary strategies, and `has_actual` marks those with an actual return. All forecasts go to one compressed Parquet file
with one row per stock-month, training history, strategy and seed (`single`, a seed number, `average` or `ensemble`).
Direct transfer does not depend on local history, so its forecasts are repeated for each history.

In [ ]:
PREDICTIONS_FILE = RESULTS_DIR / "test_predictions.parquet"
OVERWRITE_PREDICTIONS = False  # set to True only to replace completed predictions

PREDICTION_SCHEMA = pa.schema([
    ("country", pa.string()), ("id", pa.int64()), ("eom", pa.timestamp("ns")), ("predicted_month", pa.timestamp("ns")),
    ("window", pa.string()), ("strategy", pa.string()), ("seed", pa.string()), ("prediction", pa.float64()),
    ("actual_return", pa.float64()), ("has_actual", pa.bool_()), ("in_common", pa.bool_()),
])


def build_wide_table(info, strategy_predictions):
    """One row per stock-month, with actual returns and one column per strategy and seed."""
    wide = info[KEYS].copy()
    wide["actual_return"] = info[TARGET]
    wide["has_actual"] = info["has_label"]
    for strategy, seeds in strategy_predictions.items():
        for seed, predictions in seeds.items():
            column = predictions.rename(columns={"prediction": f"{strategy}|{seed}"})
            wide = wide.merge(column, on=KEYS, how="left", validate="one_to_one")
            if wide[f"{strategy}|{seed}"].isna().any():
                raise ValueError(f"{strategy} {seed}: some stock-months could not be matched.")

    wide = add_ensemble(add_seed_averages(wide))
    primary_columns = [final_column(strategy) for strategy in PRIMARY_STRATEGIES]
    wide["in_common"] = np.isfinite(wide[primary_columns]).all(axis=1)
    return wide

In [ ]:
def write_wide_table(writer, wide, window):
    """Append each strategy and seed as rows of the prediction schema."""
    for column in [name for name in wide.columns if "|" in name]:
        strategy, seed = column.split("|")
        rows = wide[KEYS + ["actual_return", "has_actual", "in_common"]].assign(
            window=window, strategy=strategy, seed=seed, prediction=wide[column].astype("float64")
        )
        table = pa.Table.from_pandas(rows[PREDICTION_SCHEMA.names], schema=PREDICTION_SCHEMA, preserve_index=False)
        writer.write_table(table)


def read_predictions(country, window, strategy, seed):
    """Read one saved forecast, ordered by month and stock."""
    selection = [("country", "==", country), ("window", "==", window), ("strategy", "==", strategy), ("seed", "==", seed)]
    df = pd.read_parquet(PREDICTIONS_FILE, filters=selection)
    return df.sort_values(["eom", "id"], ignore_index=True)

In [ ]:
def generate_predictions():
    """Predict the test period for every experiment and save one file; nothing is saved if any step fails."""
    temporary_file = PREDICTIONS_FILE.with_suffix(".tmp")
    try:
        with pq.ParquetWriter(temporary_file, PREDICTION_SCHEMA, compression="snappy") as writer:
            for country in LOCAL_COUNTRIES:
                info, X = load_test_for_inference(country)
                months = make_months(info, X)
                direct_predictions = None
                for window in TRAIN_WINDOWS[country]:
                    strategy_predictions = {}
                    for strategy in ALL_STRATEGIES:
                        if strategy == "direct" and direct_predictions is not None:
                            strategy_predictions[strategy] = direct_predictions  # same for every window
                            continue
                        print(f"Predicting: {country} | {window} | {strategy}", flush=True)
                        strategy_predictions[strategy] = predict_strategy(country, window, strategy, info, X, months)
                    direct_predictions = strategy_predictions["direct"]
                    write_wide_table(writer, build_wide_table(info, strategy_predictions), window)
        temporary_file.replace(PREDICTIONS_FILE)
    finally:
        temporary_file.unlink(missing_ok=True)

In [ ]:
if RUN_PHASE6_PREDICTIONS:
    if PREDICTIONS_FILE.exists() and not OVERWRITE_PREDICTIONS:
        print(f"Predictions: skipped ({PREDICTIONS_FILE} exists and OVERWRITE_PREDICTIONS = False)")
    else:
        print(f"Generating test predictions on {DEVICE}...")
        generate_predictions()
        print(f"Saved: {PREDICTIONS_FILE}")
else:
    print("Test predictions: not run (RUN_PHASE6_PREDICTIONS = False)")

## 32. Prediction Verification

Configuration checks always run. The saved file is checked only when predictions are enabled. No performance is
measured here.

In [ ]:
assert set(ENSEMBLE_WEIGHTS) == set(PRIMARY_STRATEGIES) and len(set(PRIMARY_STRATEGIES)) == 6
assert np.isclose(sum(ENSEMBLE_WEIGHTS.values()), 1) and len(set(ENSEMBLE_WEIGHTS.values())) == 1
assert "local" not in PRIMARY_STRATEGIES and "local" in ALL_STRATEGIES
assert PERIODS["test"] == (pd.Timestamp("2020-01-31"), pd.Timestamp("2025-12-31")) and expected_months["test"] == 72

# The training loaders must refuse the test period (they stop before reading any data)
for loader in [load_labeled, load_months]:
    try:
        loader("GBR", "test")
        test_rejected = False
    except ValueError:
        test_rejected = True
    assert test_rejected
print("Prediction configuration verified.")

In [ ]:
def verify_forecast(saved, country, window, strategy, seed):
    """Check the identifiers, dates and values of one saved forecast."""
    label = f"{country} {window} {strategy} {seed}"
    source = data[country]
    source = source[in_period(source["predicted_month"], "test")].sort_values(["eom", "id"], ignore_index=True)

    assert saved.columns.tolist() == PREDICTION_SCHEMA.names, label
    assert not saved.duplicated(KEYS).any() and len(saved) == len(source), label
    assert (saved["country"] == country).all() and (saved["id"] == source["id"]).all(), label
    for month in ["eom", "predicted_month"]:  # every stock-month is present and correctly dated
        assert (saved[month] == source[month]).all(), label
    assert np.isfinite(saved["prediction"]).all(), label
    np.testing.assert_array_equal(saved["actual_return"], source[TARGET], label)  # actual returns are unchanged
    assert (saved["has_actual"] == saved["actual_return"].notna()).all() and saved["in_common"].all(), label

In [ ]:
if RUN_PHASE6_PREDICTIONS:
    print("Verifying saved predictions...")
    for country in LOCAL_COUNTRIES:
        for window in TRAIN_WINDOWS[country]:
            values = {}
            for strategy in ALL_STRATEGIES + ["ensemble"]:
                if strategy == "ensemble":
                    seeds = ["ensemble"]
                else:
                    seeds = list(prediction_files(strategy, country, window))
                    seeds += ["average"] if strategy in TRANSFORMER_STRATEGIES else []
                for seed in seeds:
                    saved = read_predictions(country, window, strategy, seed)
                    verify_forecast(saved, country, window, strategy, seed)
                    values[strategy, seed] = saved["prediction"].to_numpy()

            # Seed averages and the ensemble are reproduced from the saved forecasts
            for strategy in TRANSFORMER_STRATEGIES:
                seed_mean = np.mean([values[strategy, str(seed)] for seed in TRANSFORMER_SEEDS], axis=0)
                assert np.allclose(values[strategy, "average"], seed_mean)
            finals = [values[tuple(final_column(strategy).split("|"))] for strategy in PRIMARY_STRATEGIES]
            assert np.allclose(values["ensemble", "ensemble"], np.mean(finals, axis=0))
            print(f"{country} | {window}: {len(values)} forecasts verified, {len(finals[0]):,} stock-months each")

    n_forecasts = len(LOCAL_COUNTRIES) * 3 * len(values)
    expected_rows = sum(in_period(data[c]["predicted_month"], "test").sum() for c in LOCAL_COUNTRIES) * 3 * len(values)
    assert pq.read_metadata(PREDICTIONS_FILE).num_rows == expected_rows
    print(f"All {n_forecasts} saved forecasts verified.")
else:
    print("Saved predictions: not verified (RUN_PHASE6_PREDICTIONS = False)")

Phase 6 — End

Phase 7 — Start

## 33. Out-of-Sample Prediction Metrics

All models are evaluated on the same stock-months of January 2020 – December 2025: those predicted by every primary
model (`in_common`) with an available actual return. Returns are never clipped or removed.

MSE is the average of 72 monthly MSE values. `OOS R² = 1 − model MSE / zero-forecast MSE`, where both use the same
stock-months and monthly averaging. Coverage is the share of stock-months with an actual return that are evaluated.

In [ ]:
MODEL_NAMES = {
    "ensemble": "Ensemble", "finetuned": "Fine-tuned Transformer", "direct": "Direct U.S. transfer",
    "joint": "Joint U.S.-local Transformer", "localmatched": "Local Transformer (matched)",
    "xgboost": "XGBoost", "elastic_net": "Elastic Net", "local": "Local Transformer (freely tuned)",
}
PRIMARY_MODELS = [name for name in MODEL_NAMES if name != "local"]
EVALUATION_DIR = RESULTS_DIR / "evaluation"

DECILE = 0.10
COST_BPS = 10  # illustrative assumption: basis points per dollar traded (the proposal gives no rate)
BLOCK_MONTHS = 6
BOOTSTRAP_SAMPLES = 2000
SIGNIFICANCE_LEVEL = 0.05


def main_seed(strategy):
    """Identifier of a model's main forecast: single model, seed average or ensemble."""
    return "ensemble" if strategy == "ensemble" else final_column(strategy).split("|")[1]


def forecast_list():
    """Every forecast to evaluate: main forecasts and individual Transformer seeds."""
    forecasts = [(strategy, main_seed(strategy)) for strategy in MODEL_NAMES]
    forecasts += [(strategy, str(seed)) for strategy in TRANSFORMER_STRATEGIES for seed in TRANSFORMER_SEEDS]
    return forecasts

In [ ]:
def load_forecast(country, window, strategy, seed):
    """Load one saved forecast and check its dates, keys and values."""
    label = f"{country} {window} {strategy} {seed}"
    stocks = read_predictions(country, window, strategy, seed)
    if stocks.empty:
        raise ValueError(f"{label}: not found in {PREDICTIONS_FILE}.")
    if stocks.duplicated(KEYS).any():
        raise ValueError(f"{label}: duplicate stock-months.")
    if not in_period(stocks["predicted_month"], "test").all() or stocks["predicted_month"].nunique() != 72:
        raise ValueError(f"{label}: dates differ from January 2020 - December 2025.")
    if not np.isfinite(stocks["prediction"]).all():
        raise ValueError(f"{label}: missing or infinite predictions.")
    if np.isinf(stocks["actual_return"]).any():
        raise ValueError(f"{label}: infinite actual returns.")
    return stocks


def squared_error_metrics(evaluated):
    """MSE of the model and of the zero forecast for one month."""
    actual = evaluated["actual_return"]
    return {"mse": ((actual - evaluated["prediction"]) ** 2).mean(), "zero_mse": (actual ** 2).mean()}

## 34. Cross-Sectional Rank Information Coefficient

The rank IC is the Spearman correlation between predicted and actual returns across stocks, calculated separately for
each month. Ties receive average ranks. A month is undefined when all predictions or all returns are equal.

In [ ]:
def rank_ic(evaluated):
    """Spearman correlation across the stocks of one month."""
    predicted, actual = evaluated["prediction"].rank(), evaluated["actual_return"].rank()
    if predicted.nunique() < 2 or actual.nunique() < 2:
        return np.nan
    return predicted.corr(actual)

## 35. Long-Short Portfolio Construction

Each month, all stocks predicted by every primary model are sorted by predicted return, with ties ordered by stock ID.
The top and bottom `floor(10% × N)` stocks form equally weighted long and short sides of $1 each.

- Stocks are selected without using actual returns. A selected stock with no actual return is left out of that side's
  average return, and the number of such stocks is recorded.
- If a model gives every stock the same forecast, it cannot rank stocks; it then takes no position and earns zero.

In [ ]:
def long_short_weights(universe):
    """Signed weights by stock ID: +1/n for the highest decile and -1/n for the lowest."""
    if universe["prediction"].nunique() < 2:
        return pd.Series(dtype="float64", index=pd.Index([], dtype="int64"))
    ids = universe.sort_values(["prediction", "id"])["id"].to_numpy()
    n = max(1, int(np.floor(DECILE * len(ids))))
    return pd.Series([-1 / n] * n + [1 / n] * n, index=np.concatenate([ids[:n], ids[-n:]]))


def portfolio_returns(universe, weights):
    """Long, short and long-short return of one month."""
    if weights.empty:
        return {"long_return": 0.0, "short_return": 0.0, "gross_return": 0.0, "n_side": 0, "missing_selected": 0}
    returns = universe.set_index("id")["actual_return"].reindex(weights.index)
    long_return, short_return = returns[weights > 0].mean(), returns[weights < 0].mean()
    if not np.isfinite([long_return, short_return]).all():
        raise ValueError("A portfolio side has no available returns.")
    return {
        "long_return": long_return, "short_return": short_return, "gross_return": long_return - short_return,
        "n_side": int((weights > 0).sum()), "missing_selected": int(returns.isna().sum()),
    }

## 36. Portfolio Performance and Transaction Costs

- **Turnover** is the traded notional per $1 long and $1 short: the sum of absolute changes in signed weights from the
  previous month. The first month includes opening the positions. Drift of weights between rebalancing dates is ignored.
- **Cost** is turnover × `COST_BPS` / 10,000. The 10 basis points are an illustrative assumption.
- **Annualization**: mean × 12, volatility and Sharpe ratio × √12.
- **Cumulative performance** is the sum of monthly returns with constant $1 sides. Returns are not compounded, because
  a long-short month can lose more than 100%. Drawdown is measured on this sum.

In [ ]:
def traded_notional(weights, previous_weights):
    """Sum of absolute weight changes, including stocks entering and leaving."""
    return weights.sub(previous_weights, fill_value=0).abs().sum()


def evaluate_forecast(stocks):
    """Monthly metrics and portfolio results of one forecast."""
    rows = []
    previous_weights = pd.Series(dtype="float64", index=pd.Index([], dtype="int64"))
    for month, month_stocks in stocks.groupby("predicted_month"):
        universe = month_stocks[month_stocks["in_common"]]
        evaluated = universe[universe["has_actual"]]
        weights = long_short_weights(universe)
        turnover = traded_notional(weights, previous_weights)
        row = {
            "predicted_month": month, "n_labeled": int(month_stocks["has_actual"].sum()), "n_evaluated": len(evaluated),
            **squared_error_metrics(evaluated), "rank_ic": rank_ic(evaluated),
            **portfolio_returns(universe, weights), "turnover": turnover, "cost": turnover * COST_BPS / 10_000,
        }
        row["net_return"] = row["gross_return"] - row["cost"]
        rows.append(row)
        previous_weights = weights
    return pd.DataFrame(rows)

In [ ]:
def sharpe_ratio(returns, axis=-1):
    """Annualized Sharpe ratio; undefined when returns do not vary."""
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.mean(returns, axis=axis) / np.std(returns, axis=axis, ddof=1) * np.sqrt(12)


def performance(returns):
    """Summary of a monthly return series."""
    cumulative = returns.cumsum()
    drawdown = cumulative - cumulative.cummax().clip(lower=0)
    return {
        "mean_monthly": returns.mean(), "annual_return": 12 * returns.mean(), "monthly_volatility": returns.std(),
        "annual_volatility": np.sqrt(12) * returns.std(), "sharpe": sharpe_ratio(returns.to_numpy()),
        "cumulative": cumulative.iloc[-1], "max_drawdown": drawdown.min(),
    }


def summarize(months):
    """Summary of one forecast from its monthly results."""
    months = months.sort_values("predicted_month")
    gross, net = performance(months["gross_return"]), performance(months["net_return"])
    return pd.Series({
        "mse": months["mse"].mean(), "oos_r2": 1 - months["mse"].mean() / months["zero_mse"].mean(),
        "stock_months": months["n_evaluated"].sum(), "months": len(months),
        "coverage": months["n_evaluated"].sum() / months["n_labeled"].sum(),
        "mean_ic": months["rank_ic"].mean(), "ic_std": months["rank_ic"].std(), "ic_months": months["rank_ic"].count(),
        **{f"gross_{name}": value for name, value in gross.items()},
        "mean_turnover": months["turnover"].mean(), "mean_cost": months["cost"].mean(),
        **{f"net_{name}": value for name, value in net.items()},
        "months_without_position": (months["n_side"] == 0).sum(), "missing_selected": months["missing_selected"].sum(),
        "months_above_100pct": (months["gross_return"].abs() > 1).sum(),
    })

In [ ]:
GROUP = ["country", "window", "strategy", "seed"]

if RUN_FINAL_EVALUATION:
    if not PREDICTIONS_FILE.exists():
        raise FileNotFoundError(f"{PREDICTIONS_FILE} is required. Run Section 31 with RUN_PHASE6_PREDICTIONS = True first.")

    results = []
    for country in LOCAL_COUNTRIES:
        for window in TRAIN_WINDOWS[country]:
            print(f"Evaluating: {country} | {window}", flush=True)
            for strategy, seed in forecast_list():
                months = evaluate_forecast(load_forecast(country, window, strategy, seed))
                results.append(months.assign(country=country, window=window, strategy=strategy, seed=seed))
    monthly = pd.concat(results, ignore_index=True)
    summary = pd.DataFrame(
        [{**dict(zip(GROUP, names)), **summarize(months)} for names, months in monthly.groupby(GROUP)]
    )
    summary["is_main"] = summary["seed"] == summary["strategy"].map(main_seed)
    summary["model"] = summary["strategy"].map(MODEL_NAMES)
else:
    print("Final evaluation: not run (RUN_FINAL_EVALUATION = False)")

In [ ]:
def main_table(columns):
    """Main forecasts in a fixed model order."""
    table = summary[summary["is_main"]].copy()
    table["order"] = table["strategy"].map(list(MODEL_NAMES).index)
    return table.sort_values(["country", "window", "order"])[["country", "window", "model"] + columns]


if RUN_FINAL_EVALUATION:
    print("Prediction accuracy and rank IC:")
    display(main_table(["mse", "oos_r2", "stock_months", "months", "coverage", "mean_ic", "ic_std", "ic_months"]))
    print("Gross long-short performance:")
    display(main_table([
        "gross_mean_monthly", "gross_annual_return", "gross_monthly_volatility", "gross_annual_volatility",
        "gross_sharpe", "gross_cumulative", "gross_max_drawdown",
    ]))
    print(f"Net of {COST_BPS} basis points per dollar traded (assumption):")
    display(main_table([
        "mean_turnover", "mean_cost", "net_mean_monthly", "net_annual_return", "net_sharpe", "net_cumulative",
        "net_max_drawdown",
    ]))

    flags = main_table(["months_without_position", "missing_selected", "months_above_100pct"])
    flags = flags[(flags[["months_without_position", "months_above_100pct"]] > 0).any(axis=1)]
    if not flags.empty:
        print("WARNING: months without a ranking, or with a long-short return beyond +/-100%. Interpret with care:")
        display(flags)

In [ ]:
if RUN_FINAL_EVALUATION:
    print("Variation across the three Transformer seeds (individual seeds, not the seed average):")
    seed_rows = summary[summary["seed"].isin([str(seed) for seed in TRANSFORMER_SEEDS])]
    seed_variation = seed_rows.groupby(["country", "window", "model"])[["oos_r2", "mean_ic", "gross_sharpe"]]
    display(seed_variation.agg(["mean", "std", "min", "max"]))

## 37. Statistical Significance Testing

Every pair of models is compared with a paired moving-block bootstrap: blocks of six consecutive months are resampled,
and the same months are used for both models. Countries and training histories are never mixed.

Differences are expressed as the **improvement of model A over model B**, so a positive value always favors A (for MSE,
the MSE of B minus the MSE of A). P-values are two-sided and use the bootstrap distribution centered at zero. Holm's
procedure is applied to the 21 primary pairs of each country, history and metric; comparisons with the freely tuned
local Transformer form a separate family. A metric undefined in any month for either model is not tested.

In [ ]:
TEST_METRICS = {  # label: (monthly column, sign of improvement, statistic)
    "MSE": ("mse", -1, np.mean), "Rank IC": ("rank_ic", 1, np.mean),
    "Gross return": ("gross_return", 1, np.mean), "Net return": ("net_return", 1, np.mean),
    "Gross Sharpe": ("gross_return", 1, sharpe_ratio), "Net Sharpe": ("net_return", 1, sharpe_ratio),
}


def block_indices(n_months, rng):
    """Month positions of every bootstrap sample, built from blocks of consecutive months."""
    n_blocks = int(np.ceil(n_months / BLOCK_MONTHS))
    starts = rng.integers(0, n_months - BLOCK_MONTHS + 1, size=(BOOTSTRAP_SAMPLES, n_blocks))
    blocks = starts[:, :, None] + np.arange(BLOCK_MONTHS)
    return blocks.reshape(BOOTSTRAP_SAMPLES, -1)[:, :n_months]


def bootstrap_difference(a, b, sign, statistic, indices):
    """Improvement of A over B with a confidence interval and two-sided p-value."""
    undefined = {"improvement": np.nan, "ci_low": np.nan, "ci_high": np.nan, "p_value": np.nan}
    if not (np.isfinite(a).all() and np.isfinite(b).all()):
        return undefined
    observed = sign * (statistic(a, axis=-1) - statistic(b, axis=-1))
    samples = sign * (statistic(a[indices], axis=-1) - statistic(b[indices], axis=-1))  # same months for both
    if not (np.isfinite(observed) and np.isfinite(samples).all()):
        return undefined

    tail = (1 - 0.95) / 2
    extreme = np.abs(samples - samples.mean()) >= np.abs(observed)  # distribution centered at zero
    return {
        "improvement": observed, "ci_low": np.quantile(samples, tail), "ci_high": np.quantile(samples, 1 - tail),
        "p_value": (1 + extreme.sum()) / (1 + len(samples)),
    }


def holm_adjust(p_values):
    """Holm-adjusted p-values; undefined tests stay undefined and are not counted."""
    valid = p_values.dropna().sort_values()
    adjusted = (valid * np.arange(len(valid), 0, -1)).cummax().clip(upper=1)
    return adjusted.reindex(p_values.index)

In [ ]:
def compare_models(monthly, country, window):
    """Paired tests of every model pair in one country and training history."""
    is_main = monthly["seed"] == monthly["strategy"].map(main_seed)
    part = monthly[is_main & (monthly["country"] == country) & (monthly["window"] == window)]
    indices = block_indices(part["predicted_month"].nunique(), np.random.default_rng(SEED))

    rows = []
    for metric, (column, sign, statistic) in TEST_METRICS.items():
        series = part.pivot(index="predicted_month", columns="strategy", values=column).sort_index()
        for a, b in combinations(MODEL_NAMES, 2):
            test = bootstrap_difference(series[a].to_numpy(), series[b].to_numpy(), sign, statistic, indices)
            family = "secondary" if "local" in (a, b) else "primary"
            rows.append({"country": country, "window": window, "metric": metric, "family": family, "a": a, "b": b, **test})
    return pd.DataFrame(rows)


def label_outcome(row):
    if pd.isna(row["p_adjusted"]):
        return "not available"
    if row["p_adjusted"] >= SIGNIFICANCE_LEVEL:
        return "no significant difference"
    return "A better" if row["improvement"] > 0 else "B better"

In [ ]:
if RUN_FINAL_EVALUATION:
    print("Running paired block-bootstrap tests...")
    comparisons = pd.concat(
        [compare_models(monthly, country, window) for country in LOCAL_COUNTRIES for window in TRAIN_WINDOWS[country]],
        ignore_index=True,
    )
    family_columns = ["country", "window", "metric", "family"]
    comparisons["p_adjusted"] = comparisons.groupby(family_columns)["p_value"].transform(holm_adjust)
    comparisons["significant"] = comparisons["p_adjusted"] < SIGNIFICANCE_LEVEL
    comparisons["outcome"] = comparisons.apply(label_outcome, axis=1)
    comparisons["model_a"] = comparisons["a"].map(MODEL_NAMES)
    comparisons["model_b"] = comparisons["b"].map(MODEL_NAMES)

    print("Primary comparisons by outcome:")
    primary = comparisons[comparisons["family"] == "primary"]
    display(primary.pivot_table(index="metric", columns="outcome", values="a", aggfunc="count", fill_value=0))

## 38. Cross-Market Model Comparisons

Each table reports the improvement of model A over model B. The matched local Transformer is the transfer benchmark.
Direct transfer is one set of U.S. models evaluated against each local history, not three separately trained models.

In [ ]:
def pair_table(pairs, metrics=("MSE", "Rank IC", "Net return")):
    """Test results of selected model pairs."""
    in_pairs = np.array([pair in pairs for pair in zip(comparisons["a"], comparisons["b"])])
    selected = comparisons[in_pairs & comparisons["metric"].isin(metrics).to_numpy()]
    return selected[[
        "country", "window", "metric", "model_a", "model_b", "improvement", "ci_low", "ci_high", "p_value",
        "p_adjusted", "significant", "outcome",
    ]]


if RUN_FINAL_EVALUATION:
    print("A. Cross-stock attention: local Transformer versus Elastic Net and XGBoost")
    display(pair_table([("localmatched", "elastic_net"), ("localmatched", "xgboost")]))
    print("B. U.S. pretraining: direct transfer and fine-tuning versus the local Transformer")
    display(pair_table([("direct", "localmatched"), ("finetuned", "localmatched")]))

In [ ]:
if RUN_FINAL_EVALUATION:
    print("C. Transfer benefit by length of local training history (improvement over the matched local Transformer)")
    transfer = pair_table([("direct", "localmatched"), ("finetuned", "localmatched"), ("joint", "localmatched")])
    display(transfer.pivot(index=["country", "metric", "model_a"], columns="window", values="improvement"))
    display(transfer.pivot(index=["country", "metric", "model_a"], columns="window", values="outcome"))

    print("D. Transfer versus pooling: fine-tuned versus joint Transformer")
    display(pair_table([("finetuned", "joint")]))

In [ ]:
if RUN_FINAL_EVALUATION:
    print("E. Ensemble versus each individual model")
    display(pair_table([("ensemble", model) for model in PRIMARY_MODELS if model != "ensemble"]))
    print("Separate comparison: matched versus freely tuned local Transformer")
    display(pair_table([("localmatched", "local")]))

## 39. Research Results and Visualizations

Essential tables, the monthly series and the figures are saved in `evaluation/` inside the results folder. Predictions
and model files are never changed.

In [ ]:
if RUN_FINAL_EVALUATION:
    EVALUATION_DIR.mkdir(parents=True, exist_ok=True)
    monthly.to_parquet(EVALUATION_DIR / "monthly_results.parquet", index=False)
    summary.to_csv(EVALUATION_DIR / "summary.csv", index=False)
    comparisons.to_csv(EVALUATION_DIR / "model_comparisons.csv", index=False)
    print(f"Saved evaluation tables in {EVALUATION_DIR}")

In [ ]:
WINDOW_NAMES = {"train_1y": "1 year", "train_3y": "3 years", "train_5y": "5 years"}


def finish_figure(figure, name):
    figure.tight_layout()
    figure.savefig(EVALUATION_DIR / f"{name}.png", dpi=200, bbox_inches="tight")
    plt.show()


def bar_figure(column, label, name, symlog=False):
    """Grouped bars by model and training history, one panel per country."""
    figure, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
    for axis, country in zip(axes, LOCAL_COUNTRIES):
        table = summary[summary["is_main"] & (summary["country"] == country)]
        table = table.pivot(index="strategy", columns="window", values=column).loc[PRIMARY_MODELS, list(WINDOW_NAMES)]
        table.rename(index=MODEL_NAMES, columns=WINDOW_NAMES).plot.barh(ax=axis)
        axis.axvline(0, color="black", linewidth=0.8)
        if symlog:
            axis.set_xscale("symlog", linthresh=0.01)  # large negative values remain visible
        axis.set(title=country, xlabel=label, ylabel="")
        axis.legend(title="Local history")
    finish_figure(figure, name)


if RUN_FINAL_EVALUATION:
    bar_figure("oos_r2", "Out-of-sample R² (symmetric log scale)", "oos_r2", symlog=True)
    bar_figure("mean_ic", "Mean monthly rank IC", "rank_ic")

In [ ]:
if RUN_FINAL_EVALUATION:
    # Cumulative gross long-short performance (sum of monthly returns, constant $1 sides)
    figure, axes = plt.subplots(2, 3, figsize=(16, 8), sharex=True)
    main_monthly = monthly[monthly["seed"] == monthly["strategy"].map(main_seed)]
    for row, country in enumerate(LOCAL_COUNTRIES):
        for column, window in enumerate(WINDOW_NAMES):
            axis = axes[row, column]
            part = main_monthly[(main_monthly["country"] == country) & (main_monthly["window"] == window)]
            series = part.pivot(index="predicted_month", columns="strategy", values="gross_return").sort_index()
            for strategy in PRIMARY_MODELS:
                axis.plot(series.index, series[strategy].cumsum(), label=MODEL_NAMES[strategy], linewidth=1.3)
            axis.axhline(0, color="black", linewidth=0.8)
            axis.set(title=f"{country}, {WINDOW_NAMES[window]}", ylabel="Sum of monthly returns")
    axes[0, 2].legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    finish_figure(figure, "cumulative_long_short")

In [ ]:
if RUN_FINAL_EVALUATION:
    # Gross versus net annualized Sharpe ratios
    figure, axes = plt.subplots(2, 3, figsize=(16, 8), sharey=True)
    for row, country in enumerate(LOCAL_COUNTRIES):
        for column, window in enumerate(WINDOW_NAMES):
            table = summary[summary["is_main"] & (summary["country"] == country) & (summary["window"] == window)]
            table = table.set_index("strategy").loc[PRIMARY_MODELS, ["gross_sharpe", "net_sharpe"]]
            table.columns = ["Gross", f"Net ({COST_BPS} bps)"]
            table.rename(index=MODEL_NAMES).plot.barh(ax=axes[row, column], legend=(row == 0 and column == 0))
            axes[row, column].axvline(0, color="black", linewidth=0.8)
            axes[row, column].set(title=f"{country}, {WINDOW_NAMES[window]}", xlabel="Annualized Sharpe ratio", ylabel="")
    finish_figure(figure, "gross_net_sharpe")

In [ ]:
if RUN_FINAL_EVALUATION:
    # Rank IC improvement over the matched local Transformer, with 95% bootstrap intervals
    figure, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
    for axis, country in zip(axes, LOCAL_COUNTRIES):
        for shift, strategy in zip([-0.1, 0, 0.1], ["direct", "finetuned", "joint"]):
            part = comparisons[
                (comparisons["country"] == country) & (comparisons["metric"] == "Rank IC")
                & (comparisons["a"] == strategy) & (comparisons["b"] == "localmatched")
            ].set_index("window").loc[list(WINDOW_NAMES)]
            errors = np.array([part["improvement"] - part["ci_low"], part["ci_high"] - part["improvement"]])
            errors = np.clip(errors, 0, None)  # a percentile interval need not contain the estimate
            positions = np.arange(len(part)) + shift
            axis.errorbar(positions, part["improvement"], yerr=errors, marker="o", capsize=4, label=MODEL_NAMES[strategy])
        axis.axhline(0, color="black", linewidth=0.8)
        axis.set_xticks(range(len(WINDOW_NAMES)), list(WINDOW_NAMES.values()))
        axis.set(title=country, xlabel="Local training history", ylabel="Rank IC improvement")
        axis.legend()
    finish_figure(figure, "transfer_benefit")

## 40. Evaluation Verification

Configuration checks always run. The small artificial checks run with `RUN_SMOKE_TESTS`, and the result checks run with
`RUN_FINAL_EVALUATION`.

In [ ]:
assert set(MODEL_NAMES) == set(ALL_STRATEGIES) | {"ensemble"} and "ensemble" not in PRIMARY_STRATEGIES
assert len(list(combinations(PRIMARY_MODELS, 2))) == 21
assert 0 < DECILE <= 0.5 and COST_BPS >= 0 and BLOCK_MONTHS == 6
assert PERIODS["validation"][1] < PERIODS["test"][0]  # selection ends before the test period
print("Evaluation configuration verified.")

In [ ]:
if RUN_SMOKE_TESTS:
    # Deciles, signs and stable ties
    example = pd.DataFrame({"id": np.arange(20), "prediction": np.arange(20) / 100, "actual_return": np.arange(20) / 10})
    weights = long_short_weights(example)
    assert set(weights[weights > 0].index) == {18, 19} and set(weights[weights < 0].index) == {0, 1}
    assert np.isclose(weights.sum(), 0) and np.isclose(weights.abs().sum(), 2)
    assert np.isclose(portfolio_returns(example, weights)["gross_return"], 1.85 - 0.05)
    assert long_short_weights(example.assign(prediction=0.0)).empty
    tied = long_short_weights(example.assign(prediction=[0.0] * 19 + [1.0]))
    assert len(tied) == 4 and tied.index.is_unique

    # Turnover: opening positions trades $2; replacing one of two longs trades 0.5 + 0.5
    nothing = weights.iloc[:0]
    changed = weights.rename(index={19: 17})
    assert np.isclose(traded_notional(weights, nothing), 2) and np.isclose(traded_notional(weights, weights), 0)
    assert np.isclose(traded_notional(changed, weights), 1) and np.isclose(traded_notional(nothing, weights), 2)

    # Rank IC is a rank correlation, and undefined without variation
    assert np.isclose(rank_ic(example.assign(actual_return=np.exp(example["actual_return"]))), 1)
    assert np.isnan(rank_ic(example.assign(prediction=0.0)))

    # Holm's procedure, and paired blocks: a constant difference has no sampling variation
    assert np.allclose(holm_adjust(pd.Series([0.01, 0.04, 0.03])), [0.03, 0.06, 0.06])
    indices = block_indices(72, np.random.default_rng(SEED))
    assert indices.shape == (BOOTSTRAP_SAMPLES, 72)
    assert (np.diff(indices.reshape(BOOTSTRAP_SAMPLES, -1, BLOCK_MONTHS), axis=2) == 1).all()
    series = np.random.default_rng(SEED).normal(size=72)
    test = bootstrap_difference(series + 0.5, series, 1, np.mean, indices)
    assert np.allclose([test["improvement"], test["ci_low"], test["ci_high"]], 0.5)
    print("Portfolio, turnover, rank IC, bootstrap and Holm calculations verified.")

In [ ]:
if RUN_FINAL_EVALUATION:
    # Dates and unique identifiers
    assert not monthly.duplicated(GROUP + ["predicted_month"]).any()
    assert monthly["predicted_month"].min() == PERIODS["test"][0] and monthly["predicted_month"].max() == PERIODS["test"][1]
    assert (monthly.groupby(GROUP).size() == 72).all()

    # Every model of an experiment is evaluated on the same stock-months and zero benchmark
    by_month = monthly.groupby(["country", "window", "predicted_month"])
    assert (by_month["n_evaluated"].nunique() == 1).all()
    assert np.allclose(by_month["zero_mse"].max(), by_month["zero_mse"].min())

    # R², portfolio signs, costs and seed labels
    average = monthly.groupby(GROUP)[["mse", "zero_mse"]].mean()
    saved_r2 = summary.set_index(GROUP)["oos_r2"].reindex(average.index)
    assert np.allclose(saved_r2, 1 - average["mse"] / average["zero_mse"])
    assert np.allclose(monthly["gross_return"], monthly["long_return"] - monthly["short_return"])
    assert np.allclose(monthly["net_return"], monthly["gross_return"] - monthly["turnover"] * COST_BPS / 10_000)
    assert monthly["turnover"].between(0, 4 + 1e-9).all()
    assert summary["is_main"].sum() == len(LOCAL_COUNTRIES) * 3 * len(MODEL_NAMES)
    assert not summary.loc[summary["is_main"], "seed"].isin([str(seed) for seed in TRANSFORMER_SEEDS]).any()

    # Multiple comparisons
    tested = comparisons.dropna(subset=["p_value"])
    assert (tested["p_adjusted"] >= tested["p_value"] - 1e-12).all() and tested["p_adjusted"].between(0, 1).all()
    assert (comparisons["significant"] == (comparisons["p_adjusted"] < SIGNIFICANCE_LEVEL)).all()
    print("Evaluation results verified.")

Phase 7 — End